# 数据分析

这份 notebook 用来把当前最小版 ADL 流程的结果，整理成一份适合机器学习实验汇报的中文分析模板。

它围绕 4 个核心问题组织：

1. 数据长什么样？
2. 模型训练得顺不顺？
3. 模型预测得准不准？
4. 模型为什么这样预测？

这份模板同时支持两条主线：

- `ADL 主线`：直接读取仓库约定的 `delta_dataset.npz`、`training_summary.json`、`uncertainty_latest.json` 和多轮 `round_*_selection_summary.json`
- `可选扩展主线`：如果你在服务器端额外准备了特征表、逐 epoch 训练 history、逐样本预测结果、可解释模型文件，就能补齐更完整的机器学习图表

适用范围说明：

- 默认按 `回归任务` 设计，主目标通常是 `delta_E`
- notebook 会自动识别最新轮次，并汇总所有历史轮次
- 当前仓库并没有跟踪真实数据、模型和结果文件，所以如果本地目录是空壳，下面很多模块会提示“缺少文件并跳过”
- 回到服务器后，只要把本 notebook 放在仓库里并保证路径设置正确，就能直接复用


In [ ]:
# 这一个代码块负责导入常用库、设置显示样式，并给后面的分析提供统一环境。
from __future__ import annotations

import importlib.util
import json
import math
import pickle
import textwrap
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython import get_ipython
from IPython.display import Markdown, display
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import MaxNLocator, PercentFormatter
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


class SimpleSeabornFallback:
    # 当环境里没有 seaborn 时，用一个轻量回退层保证 notebook 还能继续运行。
    def set_theme(self, style: str = "ticks", context: str = "notebook") -> None:  # noqa: ARG002
        if "seaborn-v0_8-whitegrid" in plt.style.available:
            plt.style.use("seaborn-v0_8-whitegrid")
        elif "seaborn-v0_8-ticks" in plt.style.available:
            plt.style.use("seaborn-v0_8-ticks")
        else:
            plt.style.use("default")

    def histplot(self, data, kde: bool = False, ax=None, color=None, bins: int = 30, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        series = pd.Series(data).dropna()
        if series.empty:
            return ax
        ax.hist(series, bins=bins, color=color, alpha=0.86, edgecolor="white", linewidth=0.9)
        if kde and len(series) > 1:
            series.plot(kind="density", ax=ax, color=color or "#243447", linewidth=1.8)
        return ax

    def boxplot(self, data=None, orient: str = "v", ax=None, color=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        if isinstance(data, pd.DataFrame):
            labels = list(data.columns)
            values = [pd.Series(data[column]).dropna().tolist() for column in labels]
            ax.boxplot(
                values,
                vert=(orient != "h"),
                labels=labels,
                patch_artist=True,
                boxprops={"facecolor": color or "#9BB9D4", "edgecolor": "#4F6A85"},
                medianprops={"color": "#B65A2A", "linewidth": 1.6},
            )
        return ax

    def heatmap(self, data, cbar: bool = True, yticklabels=True, cmap=None, annot: bool = False, square: bool = False, ax=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        matrix = np.asarray(data, dtype=float)
        image = ax.imshow(matrix, aspect="auto", cmap=cmap or "viridis")
        if cbar:
            plt.colorbar(image, ax=ax)
        if hasattr(data, "columns"):
            ax.set_xticks(range(len(data.columns)))
            ax.set_xticklabels(list(data.columns), rotation=90)
        if hasattr(data, "index") and yticklabels:
            ax.set_yticks(range(len(data.index)))
            ax.set_yticklabels(list(data.index))
        elif not yticklabels:
            ax.set_yticks([])
        if annot and matrix.size <= 400:
            for row_idx in range(matrix.shape[0]):
                for col_idx in range(matrix.shape[1]):
                    ax.text(col_idx, row_idx, f"{matrix[row_idx, col_idx]:.2f}", ha="center", va="center", fontsize=8)
        if square:
            ax.set_aspect("equal")
        return ax

    def barplot(self, data=None, x=None, y=None, hue=None, ax=None, color=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        if data is not None and hue:
            pivot = data.pivot_table(index=x, columns=hue, values=y, aggfunc="mean", fill_value=0)
            pivot.plot(kind="bar", ax=ax, color=kwargs.get("palette"))
            return ax

        if data is not None:
            x_values = data[x] if isinstance(x, str) else x
            y_values = data[y] if isinstance(y, str) else y
        else:
            x_values = x
            y_values = y

        x_series = pd.Series(x_values)
        y_series = pd.Series(y_values)
        if pd.api.types.is_numeric_dtype(x_series) and not pd.api.types.is_numeric_dtype(y_series):
            ax.barh(list(y_series), list(x_series), color=color)
        else:
            ax.bar(list(x_series), list(y_series), color=color)
        return ax

    def scatterplot(self, data=None, x=None, y=None, hue=None, ax=None, alpha: float = 1.0, s: float = 40, label=None, color=None, palette=None, **kwargs):  # noqa: ANN001, ARG002
        ax = ax or plt.gca()
        if data is not None:
            x_values = data[x] if isinstance(x, str) else x
            y_values = data[y] if isinstance(y, str) else y
        else:
            x_values = x
            y_values = y

        if hue is not None and data is not None:
            hue_values = data[hue] if isinstance(hue, str) else hue
            unique_hues = list(pd.Series(hue_values).astype(str).fillna("未标注").unique())
            palette = palette or ["#24476B", "#C77A43", "#5E6D7D", "#4C7C7D", "#A56A7D"]
            for idx, hue_name in enumerate(unique_hues):
                mask = pd.Series(hue_values).astype(str).fillna("未标注") == hue_name
                ax.scatter(
                    pd.Series(x_values)[mask],
                    pd.Series(y_values)[mask],
                    alpha=alpha,
                    s=s,
                    label=str(hue_name),
                    color=palette[idx % len(palette)],
                    edgecolor="white",
                    linewidth=0.6,
                )
            return ax

        ax.scatter(x_values, y_values, alpha=alpha, s=s, label=label, color=color, edgecolor="white", linewidth=0.6)
        return ax


try:
    import seaborn as sns

    HAS_SEABORN = True
except ModuleNotFoundError:
    sns = SimpleSeabornFallback()
    HAS_SEABORN = False
    print("提示：当前环境未安装 seaborn，下面会自动回退到 matplotlib 风格，图能继续画，但样式会更朴素。")


warnings.filterwarnings("ignore")
pd.options.display.max_columns = 200
pd.options.display.max_rows = 200


ipython = get_ipython()
if ipython is not None:
    try:
        ipython.run_line_magic("config", "InlineBackend.figure_format = 'retina'")
    except Exception:
        pass


THEME_PRESETS = {
    "paper_blue_gray": {
        "display_name": "论文插图风（蓝灰克制）",
        "colors": {
            "navy": "#24476B",
            "blue": "#5F86A8",
            "teal": "#4C7C7D",
            "orange": "#C77A43",
            "gold": "#BFA15C",
            "rose": "#A56A7D",
            "slate": "#5E6D7D",
            "ink": "#22313F",
            "grid": "#D5DEE7",
            "panel": "#F5F7FA",
            "paper": "#FFFFFF",
        },
        "heatmap": ["#F8FBFF", "#DCE6F0", "#89A6C7", "#24476B"],
        "diverging": ["#355C85", "#C9D7E5", "#F8F8F8", "#F0D3C1", "#B76732"],
        "table_gradient": ["#F4F7FB", "#D7E2EE", "#5F86A8"],
        "table_header": "#24476B",
    },
    "presentation_high_contrast": {
        "display_name": "答辩汇报风（高对比）",
        "colors": {
            "navy": "#103C68",
            "blue": "#1F77B4",
            "teal": "#0E8A8A",
            "orange": "#D55E00",
            "gold": "#B8860B",
            "rose": "#C44E52",
            "slate": "#3E4A59",
            "ink": "#17212B",
            "grid": "#CBD5E1",
            "panel": "#FCFDFF",
            "paper": "#FFFFFF",
        },
        "heatmap": ["#F8FBFF", "#CFE4F8", "#5B9BD5", "#103C68"],
        "diverging": ["#175A94", "#CBE0F5", "#FFFFFF", "#F9D5C5", "#C44900"],
        "table_gradient": ["#F8FBFF", "#CFE4F8", "#1F77B4"],
        "table_header": "#103C68",
    },
}


REPORT_COLORS: dict[str, str] = {}
REPORT_PALETTE: list[str] = []
REPORT_HEATMAP_CMAP = None
REPORT_DIVERGING_CMAP = None
REPORT_TABLE_CMAP = None
ACTIVE_THEME_NAME = None

SPLIT_DISPLAY_MAP = {
    "subtrain": "训练子集",
    "validation": "验证子集",
    "overall": "总体",
}

FIGURE_SIZE_PRESETS = {
    "standard": (8.0, 6.0),
    "wide": (10.8, 5.8),
    "square": (6.8, 6.8),
    "tall": (8.2, 7.4),
    "hero": (12.4, 6.4),
}

PRETTY_LABELS = {
    "sample_id": "样本编号",
    "split": "数据切分",
    "split_display": "数据切分",
    "subtrain": "训练子集",
    "validation": "验证子集",
    "overall": "总体",
    "E_baseline": "基线能量",
    "E_target": "目标能量",
    "delta_E": "Delta 能量 ΔE",
    "|F_baseline|": "基线力范数",
    "|F_target|": "目标力范数",
    "|delta_F|": "力差范数 ΔF",
    "num_atoms": "原子数",
    "charge": "电荷",
    "multiplicity": "自旋多重度",
    "source": "来源",
    "geometry_file": "几何文件",
    "mean_atomic_number": "平均原子序数",
    "coordinate_std": "坐标标准差",
    "y_true": "真实值",
    "y_pred": "预测值",
    "residual": "残差",
    "abs_error": "绝对误差",
    "uncertainty": "不确定性",
    "loss": "损失值",
    "rmse": "均方根误差 RMSE",
    "mae": "平均绝对误差 MAE",
    "mse": "均方误差 MSE",
    "accuracy": "准确率",
    "R2": "决定系数 R²",
    "Epoch": "训练轮次",
    "epoch": "训练轮次",
    "true_value_bin": "真实值区间",
    "sample_count": "样本数",
    "mean_abs_error": "平均绝对误差",
    "bin_rmse": "区间 RMSE",
    "feature": "特征",
    "importance": "重要性",
    "round_index": "轮次编号",
    "latest_round_index": "最新轮次",
    "current_round_index": "当前分析轮次",
    "total_rounds": "累计轮次数",
    "selected_count": "本轮新增样本数",
    "summary_selected_count": "汇总中记录的新增样本数",
    "manifest_selected_count": "manifest 记录的新增样本数",
    "selected_manifest_exists": "是否存在选点 manifest",
    "selected_count_consistent": "选点数量是否一致",
    "num_pool_samples": "几何池样本数",
    "num_uncertain_samples": "高不确定性样本数",
    "uncertain_ratio": "高不确定性比例",
    "converged": "是否收敛",
    "current_labeled_samples": "当前已标注样本数",
    "remaining_candidate_samples": "当前剩余候选池样本数",
    "labeled_samples_before_selection": "该轮选点前已标注样本数",
    "expected_labeled_after_selection": "该轮选点后预期样本数",
    "selection_summary_file": "选点汇总文件",
    "selection_manifest_file": "选点 manifest 文件",
    "updated_at": "更新时间",
}

TOKEN_LABELS = {
    "energy": "能量",
    "gradient": "梯度",
    "force": "力",
    "rmse": "RMSE",
    "mae": "MAE",
    "mse": "MSE",
    "loss": "损失值",
    "accuracy": "准确率",
    "pcc": "PCC",
    "error": "误差",
    "round": "轮次",
    "selected": "选点",
    "uncertain": "高不确定性",
    "ratio": "比例",
    "count": "数量",
}


def get_config_value(key: str, default: Any) -> Any:
    config = globals().get("CONFIG", {})
    if isinstance(config, dict):
        return config.get(key, default)
    return default


def to_cn_label(label: Any) -> str:
    # 把常见英文列名、指标名尽量转成中文，保证图里的横纵坐标更适合汇报。
    if label is None:
        return ""
    text = str(label)
    if text in PRETTY_LABELS:
        return PRETTY_LABELS[text]

    for prefix, cn_prefix in [
        ("train_", "训练集"),
        ("validation_", "验证集"),
        ("valid_", "验证集"),
        ("val_", "验证集"),
        ("subtrain_", "训练子集"),
        ("overall_", "总体"),
    ]:
        if text.startswith(prefix):
            return cn_prefix + to_cn_label(text[len(prefix):])

    tokens = text.split("_")
    if any(token in TOKEN_LABELS or token in PRETTY_LABELS for token in tokens):
        return "".join(TOKEN_LABELS.get(token, PRETTY_LABELS.get(token, token)) for token in tokens)
    return text.replace("_", " ")



CN_FONT_CANDIDATES = [
    "Microsoft YaHei",
    "SimHei",
    "Noto Sans CJK SC",
    "WenQuanYi Zen Hei",
    "Arial Unicode MS",
    "DejaVu Sans",
]


def resolve_font_candidates() -> tuple[list[str], list[str]]:
    # 中文字体候选顺序统一放在这里，优先命中 Windows / Linux 常见字体。
    try:
        installed_fonts = {entry.name for entry in font_manager.fontManager.ttflist}
    except Exception:
        installed_fonts = set()
    available = [name for name in CN_FONT_CANDIDATES if name in installed_fonts]
    missing = [name for name in CN_FONT_CANDIDATES if name not in available]
    return available, missing


AVAILABLE_CN_FONTS, MISSING_CN_FONTS = resolve_font_candidates()
PLOT_FONT_SANS_SERIF = AVAILABLE_CN_FONTS + MISSING_CN_FONTS


def build_font_environment_note() -> str:
    # 如果本机没装候选字体，就明确提醒后续可能回退到不理想字体。
    if not AVAILABLE_CN_FONTS:
        return "环境备注：当前环境未检测到候选中文字体，matplotlib 会回退到系统默认 sans-serif，中文仍可能显示为方块或乱码。"
    if not MISSING_CN_FONTS:
        return "环境备注：当前候选中文字体均已可用。"
    return "环境备注：若本机未安装这些候选字体，matplotlib 仍会回退到系统 sans-serif，中文显示可能不理想。缺失字体：" + ", ".join(MISSING_CN_FONTS)


def shorten_label_text(text: Any, max_chars: int = 28) -> str:
    # ??????????????????????????????
    value = str(text).replace("\n", " ").strip()
    if len(value) <= max_chars:
        return value
    return value[: max_chars - 3] + "..."


def wrap_cn_text(text: Any, width: int = 10, max_chars: int = 28) -> str:
    # ???????????????? notebook ?????????
    normalized = shorten_label_text(to_cn_label(text), max_chars=max_chars)
    return textwrap.fill(normalized, width=width, break_long_words=False, break_on_hyphens=False)


def estimate_barh_height(n_items: int, base: float = 3.8, step: float = 0.42, max_height: float = 11.5) -> float:
    # ???????????????????????????
    return min(max_height, max(base, base + max(n_items - 1, 0) * step))


def create_figure(
    kind: str = "standard",
    *,
    width_scale: float = 1.0,
    height_scale: float = 1.0,
    figsize: tuple[float, float] | None = None,
):
    # Central figure factory for notebook-friendly layouts.
    if figsize is None:
        width, height = FIGURE_SIZE_PRESETS.get(kind, FIGURE_SIZE_PRESETS["standard"])
        figsize = (width * width_scale, height * height_scale)
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(REPORT_COLORS.get("paper", "#FFFFFF"))
    return fig, ax


def create_subplots(
    nrows: int,
    ncols: int,
    *,
    kind: str = "standard",
    sharex: bool = False,
    sharey: bool = False,
    width_scale: float = 1.0,
    height_scale: float = 1.0,
    figsize: tuple[float, float] | None = None,
):
    # Central subplot factory tuned for notebook viewing and screenshots.
    if figsize is None:
        width, height = FIGURE_SIZE_PRESETS.get(kind, FIGURE_SIZE_PRESETS["standard"])
        width = width * max(1.0, 0.62 + ncols * 0.38) * width_scale
        height = height * max(1.0, 0.74 + nrows * 0.38) * height_scale
        figsize = (width, height)
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, sharex=sharex, sharey=sharey)
    fig.patch.set_facecolor(REPORT_COLORS.get("paper", "#FFFFFF"))
    return fig, axes


def create_heatmap_figure(
    n_rows: int,
    n_cols: int,
    *,
    min_width: float = 8.8,
    max_width: float = 15.2,
    min_height: float = 5.2,
    max_height: float = 11.2,
    width_per_col: float = 0.36,
    height_per_row: float = 0.04,
):
    # Heatmaps need a different sizing curve to stay legible with many labels.
    width = min(max_width, max(min_width, min_width + max(n_cols - 1, 0) * width_per_col))
    height = min(max_height, max(min_height, min_height + max(n_rows - 1, 0) * height_per_row))
    return create_figure(figsize=(width, height))


def style_axis(
    ax,
    title: str,
    xlabel: str | None = None,
    ylabel: str | None = None,
    *,
    grid_axis: str = "y",
) -> None:
    # ?????????????????????????
    ax.set_title(title, loc="left", pad=12, color=REPORT_COLORS["ink"], fontweight="bold", fontsize=15)
    if xlabel is not None:
        ax.set_xlabel(xlabel, labelpad=8, color=REPORT_COLORS["ink"])
    if ylabel is not None:
        ax.set_ylabel(ylabel, labelpad=8, color=REPORT_COLORS["ink"])
    ax.set_facecolor(REPORT_COLORS["panel"])
    ax.set_axisbelow(True)
    ax.grid(False)
    if grid_axis in {"x", "both"}:
        ax.grid(axis="x", color=REPORT_COLORS["grid"], linestyle="--", linewidth=0.8, alpha=0.36)
    if grid_axis in {"y", "both"}:
        ax.grid(axis="y", color=REPORT_COLORS["grid"], linestyle="--", linewidth=0.8, alpha=0.36)
    ax.tick_params(axis="x", pad=5)
    ax.tick_params(axis="y", pad=5)
    ax.margins(x=0.03, y=0.05)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_color(REPORT_COLORS["grid"])
        ax.spines[spine].set_linewidth(1.0)


def resolve_tick_label_fontsize(label_count: int, longest: int, *, axis: str = "x") -> float:
    # Keep dense notebook labels readable without forcing every plot to set a font size manually.
    if axis == "x":
        if label_count >= 10 or longest >= 18:
            return 8.5
        if label_count >= 6 or longest >= 12:
            return 9.0
        return 10.0
    if label_count >= 12 or longest >= 22:
        return 8.6
    if label_count >= 7 or longest >= 16:
        return 9.1
    return 9.8


def clean_long_labels(
    labels_or_ax,
    *,
    axis: str = "x",
    rotation: int | None = None,
    width: int = 12,
    ha: str = "right",
    max_chars: int = 28,
    font_size: float | None = None,
    max_ticks: int | None = None,
):
    # Long labels are wrapped, truncated, and optionally sparsified in one place.
    def _format_label(text: Any) -> str:
        return wrap_cn_text(text, width=width, max_chars=max_chars)

    def _sparsify(labels: list[str], keep_count: int | None) -> list[str]:
        if keep_count is None or keep_count <= 0 or len(labels) <= keep_count:
            return labels
        keep_positions = {0, len(labels) - 1}
        if keep_count > 2:
            step = max(1, int(round((len(labels) - 1) / float(keep_count - 1))))
            keep_positions.update(range(0, len(labels), step))
            keep_positions.add(len(labels) - 1)
        return [label if idx in keep_positions else "" for idx, label in enumerate(labels)]

    if hasattr(labels_or_ax, "get_xticklabels"):
        ax = labels_or_ax
        is_x_axis = axis == "x"
        tick_labels = ax.get_xticklabels() if is_x_axis else ax.get_yticklabels()
        raw_texts = [label.get_text() for label in tick_labels]
        if not raw_texts:
            return []
        formatted = [_format_label(text) for text in raw_texts]
        longest = max((len(str(text).replace("\n", "")) for text in formatted), default=0)
        if rotation is None:
            rotation = 28 if is_x_axis and (len(formatted) >= 6 or longest >= 10) else 0
        effective_font_size = font_size or resolve_tick_label_fontsize(len(formatted), longest, axis=axis)

        if is_x_axis:
            effective_max_ticks = max_ticks if max_ticks is not None else int(get_config_value("MAX_X_TICK_LABELS", 16))
            formatted = _sparsify(formatted, effective_max_ticks)
            ax.set_xticklabels(formatted, rotation=rotation, ha=ha if rotation else "center", fontsize=effective_font_size)
            ax.tick_params(axis="x", labelsize=effective_font_size)
        else:
            ax.set_yticklabels(formatted, rotation=rotation, fontsize=effective_font_size)
            ax.tick_params(axis="y", labelsize=effective_font_size)
        return formatted

    if isinstance(labels_or_ax, (str, bytes)):
        return _format_label(labels_or_ax)

    try:
        return [_format_label(text) for text in labels_or_ax]
    except TypeError:
        return _format_label(labels_or_ax)

def tidy_tick_labels(
    ax,
    axis: str = "x",
    rotation: int | None = None,
    width: int | None = None,
    ha: str = "right",
    max_chars: int = 28,
) -> None:
    # Backward-compatible wrapper around clean_long_labels().
    clean_long_labels(
        ax,
        axis=axis,
        rotation=rotation,
        width=width or 12,
        ha=ha,
        max_chars=max_chars,
        max_ticks=int(get_config_value("MAX_X_TICK_LABELS", 16)) if axis == "x" else None,
    )

def build_palette(size: int) -> list[str]:
    # 根据主题色板长度循环取色。
    return [REPORT_PALETTE[idx % len(REPORT_PALETTE)] for idx in range(size)]


def build_filtered_note(skipped_items: list[str], *, label: str = "字段", limit: int = 5) -> str | None:
    # 统一汇总被过滤掉的低信息量内容，避免输出里充满噪音。
    if not skipped_items:
        return None
    preview = ", ".join(to_cn_label(item) for item in skipped_items[:limit])
    if len(skipped_items) > limit:
        preview += " 等"
    return f"已跳过 {len(skipped_items)} 个低信息量{label}：{preview}。"


def is_informative_numeric_series(series: pd.Series, *, min_non_null: int = 2) -> bool:
    # 过滤全空、全零、全相同等数值列。
    clean = pd.to_numeric(pd.Series(series), errors="coerce").dropna()
    if len(clean) < min_non_null:
        return False
    if clean.nunique(dropna=True) <= 1:
        return False
    return not bool(np.isclose(clean.abs().sum(), 0.0))


def select_informative_numeric_columns(
    frame: pd.DataFrame,
    numeric_cols: list[str],
    *,
    max_cols: int | None = None,
) -> tuple[list[str], list[str]]:
    kept: list[str] = []
    skipped: list[str] = []
    for column in numeric_cols:
        if column not in frame.columns:
            continue
        if is_informative_numeric_series(frame[column]):
            kept.append(column)
        else:
            skipped.append(column)
    if max_cols is not None and len(kept) > max_cols:
        skipped.extend(kept[max_cols:])
        kept = kept[:max_cols]
    return kept, skipped


def is_informative_categorical_series(series: pd.Series, *, min_non_null: int = 2) -> bool:
    # 过滤只有一个类别或只有零星样本的类别列。
    clean = pd.Series(series).dropna().astype(str).str.strip()
    clean = clean[clean != ""]
    if len(clean) < min_non_null:
        return False
    return clean.nunique(dropna=True) > 1


def build_category_counts(
    series: pd.Series,
    *,
    top_n: int = 8,
    min_count: int = 2,
) -> tuple[pd.Series, str | None]:
    # 分类展示只保留有频次支撑的类别，并默认截断长尾。
    clean = pd.Series(series).fillna("未标注").astype(str).str.strip()
    clean = clean[clean != ""]
    if len(clean) < 2:
        return pd.Series(dtype=int), "非空样本过少"
    raw_counts = clean.value_counts()
    if raw_counts.index.nunique() <= 1:
        return pd.Series(dtype=int), "类别只有 1 种"
    informative = raw_counts[raw_counts >= min_count]
    if informative.empty:
        return pd.Series(dtype=int), f"所有类别样本数都小于 {min_count}"
    note = None
    if len(informative) > top_n:
        note = f"仅展示前 {top_n} 个高频类别"
    return informative.head(top_n).sort_values(ascending=True), note


def is_informative_line_series(values: list[float], *, min_points: int = 2) -> bool:
    # 折线图至少要有两个点且存在变化。
    series = pd.Series(values, dtype=float).dropna()
    return len(series) >= min_points and series.nunique(dropna=True) > 1


def format_percent_axis(ax, axis: str = "y", decimals: int = 0) -> None:
    # ? 0-1 ???????????????
    formatter = PercentFormatter(xmax=1.0, decimals=decimals)
    if axis == "x":
        ax.xaxis.set_major_formatter(formatter)
    else:
        ax.yaxis.set_major_formatter(formatter)


def set_integer_ticks(ax, axis: str = "y") -> None:
    # ???????????????
    locator = MaxNLocator(integer=True)
    if axis == "x":
        ax.xaxis.set_major_locator(locator)
    else:
        ax.yaxis.set_major_locator(locator)


def expand_axis_limit_for_labels(
    ax,
    data_max: float | None = None,
    *,
    orientation: str = "vertical",
    margin_ratio: float = 0.15,
) -> None:
    # ???????????????????????????
    if orientation == "horizontal":
        axis_min, axis_max = ax.get_xlim()
    else:
        axis_min, axis_max = ax.get_ylim()
    axis_span = max(abs(axis_max - axis_min), 1e-8)
    target_max = axis_max
    if data_max is not None and np.isfinite(data_max):
        target_max = max(target_max, float(data_max))
    expanded_max = target_max + axis_span * margin_ratio
    if orientation == "horizontal":
        ax.set_xlim(axis_min, expanded_max)
    else:
        ax.set_ylim(axis_min, expanded_max)


def style_colorbar(colorbar, label: str | None = None) -> None:
    # ?? colorbar ??????????????
    if colorbar is None:
        return
    if label:
        colorbar.set_label(label, color=REPORT_COLORS["ink"], fontsize=10.5)
    colorbar.ax.tick_params(labelsize=9, colors=REPORT_COLORS["ink"])
    if hasattr(colorbar, "outline") and colorbar.outline is not None:
        colorbar.outline.set_edgecolor(REPORT_COLORS["grid"])
        colorbar.outline.set_linewidth(0.8)


def finalize_figure(fig, *, rect: tuple[float, float, float, float] | None = None) -> None:
    # Figure layout is finalized here so plots stay compact and screenshot-friendly.
    fig.patch.set_facecolor(REPORT_COLORS.get("paper", "#FFFFFF"))
    default_rect = (0.02, 0.04, 0.98, 0.95 if getattr(fig, "_suptitle", None) is not None else 0.98)
    try:
        fig.tight_layout(rect=rect or default_rect, pad=1.04)
    except Exception:
        fig.subplots_adjust(left=0.10, right=0.98, bottom=0.12, top=0.94)


def set_figure_title(fig, title: str | None = None, *, y: float = 1.02) -> None:
    # suptitle styling stays centralized to avoid drift across plots.
    if not title:
        return
    fig.suptitle(title, x=0.05, y=y, ha="left", fontsize=15, fontweight="bold", color=REPORT_COLORS["ink"])


def finalize_plot(
    fig,
    *,
    title: str | None = None,
    rect: tuple[float, float, float, float] | None = None,
    title_y: float = 1.02,
    show: bool = True,
) -> None:
    # Unified plot finalizer: title, layout, despine, then display.
    set_figure_title(fig, title=title, y=title_y)
    finalize_figure(fig, rect=rect)
    if HAS_SEABORN and hasattr(sns, "despine"):
        try:
            sns.despine(fig=fig, trim=True)
        except Exception:
            pass
    if show:
        plt.show()


def create_barh_figure(
    n_items: int,
    *,
    width: float = 10.8,
    base: float = 3.8,
    step: float = 0.42,
    max_height: float = 9.8,
):
    # 横向条形图统一按照条目数自动扩展高度。
    fig, ax = plt.subplots(figsize=(width, estimate_barh_height(n_items, base=base, step=step, max_height=max_height)))
    fig.patch.set_facecolor(REPORT_COLORS.get("paper", "#FFFFFF"))
    return fig, ax


def format_bar_label_value(
    value: float,
    *,
    fmt: str | None = None,
    decimals: int = 2,
    as_percent: bool = False,
) -> str:
    # 柱状图标注值统一在这里格式化，避免百分比和小数位各写一套逻辑。
    if fmt is not None:
        return fmt.format(value)
    if as_percent:
        return f"{value:.{decimals}%}"
    return f"{value:.{decimals}f}"


def annotate_bars(
    ax,
    fmt: str | None = None,
    skip_zero: bool = False,
    *,
    orientation: str = "vertical",
    decimals: int = 2,
    as_percent: bool = False,
) -> None:
    # 柱状图数值标注统一放这里，明确按方向读取真实数据值，避免把默认柱宽 0.8 当成标注值。
    patches = list(ax.patches)
    if not patches:
        return
    if orientation not in {"vertical", "horizontal"}:
        raise ValueError("orientation must be 'vertical' or 'horizontal'")

    axis_min, axis_max = ax.get_ylim() if orientation == "vertical" else ax.get_xlim()
    axis_span = max(abs(axis_max - axis_min), 1e-8)
    pad = axis_span * 0.015

    for patch in patches:
        value = patch.get_height() if orientation == "vertical" else patch.get_width()
        if not np.isfinite(value):
            continue
        if skip_zero and abs(value) < 1e-12:
            continue

        label_text = format_bar_label_value(value, fmt=fmt, decimals=decimals, as_percent=as_percent)

        if orientation == "vertical":
            x_pos = patch.get_x() + patch.get_width() / 2
            y_pos = value + pad if value >= 0 else value - pad
            ax.text(
                x_pos,
                y_pos,
                label_text,
                ha="center",
                va="bottom" if value >= 0 else "top",
                fontsize=9,
                color=REPORT_COLORS["slate"],
                clip_on=False,
            )
        else:
            x_pos = value + pad if value >= 0 else value - pad
            y_pos = patch.get_y() + patch.get_height() / 2
            ax.text(
                x_pos,
                y_pos,
                label_text,
                ha="left" if value >= 0 else "right",
                va="center",
                fontsize=9,
                color=REPORT_COLORS["slate"],
                clip_on=False,
            )


def add_bar_labels(
    ax,
    fmt: str | None = None,
    skip_zero: bool = False,
    *,
    orientation: str = "vertical",
    decimals: int = 2,
    as_percent: bool = False,
) -> None:
    # 兼容旧调用；新的柱状图标注统一改走 annotate_bars()。
    annotate_bars(
        ax,
        fmt=fmt,
        skip_zero=skip_zero,
        orientation=orientation,
        decimals=decimals,
        as_percent=as_percent,
    )


def prepare_table_frame(frame: pd.DataFrame, rename_columns: bool = False) -> pd.DataFrame:
    table_df = frame.copy()

    if rename_columns:
        table_df.columns = [to_cn_label(column) for column in table_df.columns]

    for column in ["??", "feature", "???", "???", "split", "split_display"]:
        if column in table_df.columns:
            table_df[column] = table_df[column].map(to_cn_label)

    # Prioritize key report columns first, then keep the rest stable.
    priority_keywords = [
        "??", "??", "split", "??", "??", "??", "??", "uncertainty", "??", "??", "??", "feature", "??",
    ]

    def score(column: str) -> tuple[int, int, str]:
        text = str(column).lower()
        hits = sum(1 for key in priority_keywords if key.lower() in text)
        return (-hits, len(text), text)

    table_df = table_df.loc[:, ~table_df.columns.duplicated()].copy()
    ordered_cols = sorted(table_df.columns.tolist(), key=score)
    return table_df[ordered_cols]

def display_table(
    frame: pd.DataFrame,
    title: str | None = None,
    rename_columns: bool = False,
    precision: int | None = None,
    max_rows: int | None = None,
    gradient_subset: list[str] | None = None,
) -> None:
    # 给 DataFrame 套上统一的汇报风格，避免表格和图的视觉语言脱节。
    if frame is None or frame.empty:
        if title:
            display(Markdown(f"### {title}"))
        print("当前没有可展示的表格内容。")
        return

    if title:
        display(Markdown(f"### {title}"))

    preview = prepare_table_frame(frame, rename_columns=rename_columns)
    max_rows = max_rows or get_config_value("TABLE_MAX_ROWS", 30)
    precision = precision if precision is not None else get_config_value("TABLE_PRECISION", 4)
    truncated = len(preview) > max_rows
    preview = preview.head(max_rows).copy()

    numeric_cols = preview.select_dtypes(include=[np.number]).columns.tolist()
    subset = gradient_subset
    if subset is None:
        subset = numeric_cols[: min(4, len(numeric_cols))]
    subset = [column for column in subset if column in preview.columns]

    styler = preview.style.hide(axis="index")
    styler = styler.format(precision=precision, na_rep="—")
    styler = styler.set_properties(
        **{
            "border": f"1px solid {REPORT_COLORS['grid']}",
            "padding": "8px 10px",
            "font-size": "12px",
            "white-space": "normal",
            "max-width": "360px",
            "color": REPORT_COLORS["ink"],
        }
    )
    styler = styler.set_table_styles(
        [
            {
                "selector": "th.col_heading",
                "props": [
                    ("background-color", THEME_PRESETS[ACTIVE_THEME_NAME]["table_header"]),
                    ("color", "#FFFFFF"),
                    ("font-weight", "bold"),
                    ("border", f"1px solid {REPORT_COLORS['grid']}"),
                    ("padding", "8px 10px"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("background-color", REPORT_COLORS["paper"]),
                ],
            },
            {
                "selector": "tbody tr:nth-child(even) td",
                "props": [
                    ("background-color", REPORT_COLORS["panel"]),
                ],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("margin", "6px 0 14px 0"),
                ],
            },
        ]
    )
    if subset:
        styler = styler.background_gradient(cmap=REPORT_TABLE_CMAP, subset=subset)
    display(styler)
    if truncated:
        print(f"提示：该表原始共有 {len(frame)} 行，这里只展示前 {max_rows} 行。")


DEFAULT_STYLE_PRESET = "paper_blue_gray"
DEFAULT_SEABORN_STYLE = "ticks"
DEFAULT_SEABORN_CONTEXT = "notebook"
DEFAULT_FIGURE_SIZE = (8.2, 6.0)
DEFAULT_FIGURE_DPI = 140
DEFAULT_SAVEFIG_DPI = 160


def resolve_theme_name(theme_name: str | None = None) -> str:
    # ?????????? notebook ?????????????
    candidate = theme_name or DEFAULT_STYLE_PRESET
    return candidate if candidate in THEME_PRESETS else DEFAULT_STYLE_PRESET


def apply_theme_tokens(theme_name: str | None = None) -> None:
    # ???????? colormap???? seaborn / rcParams ???????
    global ACTIVE_THEME_NAME, REPORT_HEATMAP_CMAP, REPORT_DIVERGING_CMAP, REPORT_TABLE_CMAP
    theme_key = resolve_theme_name(theme_name)
    ACTIVE_THEME_NAME = theme_key
    REPORT_COLORS.clear()
    REPORT_COLORS.update(THEME_PRESETS[theme_key]["colors"])

    REPORT_PALETTE.clear()
    REPORT_PALETTE.extend(
        [
            REPORT_COLORS["navy"],
            REPORT_COLORS["teal"],
            REPORT_COLORS["orange"],
            REPORT_COLORS["rose"],
            REPORT_COLORS["gold"],
            REPORT_COLORS["blue"],
        ]
    )
    REPORT_HEATMAP_CMAP = LinearSegmentedColormap.from_list("report_heatmap", THEME_PRESETS[theme_key]["heatmap"])
    REPORT_DIVERGING_CMAP = LinearSegmentedColormap.from_list("report_diverging", THEME_PRESETS[theme_key]["diverging"])
    REPORT_TABLE_CMAP = LinearSegmentedColormap.from_list("report_table", THEME_PRESETS[theme_key]["table_gradient"])


def build_plot_rc_params() -> dict[str, Any]:
    # 中文字体、dpi、默认尺寸只在这里统一配置，后续 cell 不要再单独覆盖。
    return {
        "font.family": "sans-serif",
        "font.sans-serif": PLOT_FONT_SANS_SERIF,
        "axes.unicode_minus": False,
        "figure.figsize": DEFAULT_FIGURE_SIZE,
        "figure.dpi": DEFAULT_FIGURE_DPI,
        "savefig.dpi": DEFAULT_SAVEFIG_DPI,
        "savefig.bbox": "tight",
        "figure.facecolor": REPORT_COLORS["paper"],
        "axes.facecolor": REPORT_COLORS["panel"],
        "axes.edgecolor": REPORT_COLORS["grid"],
        "axes.labelcolor": REPORT_COLORS["ink"],
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 11.5,
        "axes.linewidth": 1.15,
        "axes.axisbelow": True,
        "xtick.color": REPORT_COLORS["ink"],
        "ytick.color": REPORT_COLORS["ink"],
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "xtick.major.pad": 6,
        "ytick.major.pad": 6,
        "grid.color": REPORT_COLORS["grid"],
        "grid.alpha": 0.32,
        "grid.linestyle": "--",
        "grid.linewidth": 0.8,
        "legend.frameon": False,
        "legend.fontsize": 10.5,
        "legend.title_fontsize": 10.5,
        "lines.linewidth": 2.0,
        "lines.markersize": 6,
    }


def setup_plot_style(
    theme_name: str | None = None,
    *,
    seaborn_style: str = DEFAULT_SEABORN_STYLE,
    seaborn_context: str = DEFAULT_SEABORN_CONTEXT,
) -> None:
    # ??????????? seaborn theme?? matplotlib rcParams?
    apply_theme_tokens(theme_name)
    sns.set_theme(style=seaborn_style, context=seaborn_context)
    if HAS_SEABORN and hasattr(sns, "set_palette"):
        sns.set_palette(REPORT_PALETTE)
    plt.rcParams.update(build_plot_rc_params())


setup_plot_style(DEFAULT_STYLE_PRESET)


print("环境初始化完成。")
print(f"retina 模式可用：{ipython is not None}")
print(f"seaborn 可用：{HAS_SEABORN}")
print(f"当前默认视觉预设：{THEME_PRESETS[ACTIVE_THEME_NAME]['display_name']} ({ACTIVE_THEME_NAME})")
print("可选视觉预设：paper_blue_gray / presentation_high_contrast")
print("中文字体候选顺序：", " -> ".join(CN_FONT_CANDIDATES))
print("当前 matplotlib sans-serif 字体栈：", " -> ".join(PLOT_FONT_SANS_SERIF))
print(build_font_environment_note())
print("如果想切换整本 notebook 的绘图风格，请修改 DEFAULT_STYLE_PRESET 后重跑本 cell。")
print("这一步说明什么：后面所有图和表都会复用这里的中文标签、统一尺寸、中文字体回退和更适合科研汇报的绘图风格。")


## 0. 配置区

这一节只改最上面的参数，不要急着改后面的分析逻辑。

推荐使用方式：

- 如果你只想分析当前仓库默认结果，先保持外部路径为 `None`
- 如果你在服务器上还有额外文件，就把对应路径填进去
- 如果你的目标列不是 `delta_E`，请优先修改 `TARGET_COL`
- 如果你想自动跟踪最新轮次，保持 `ROUND_SELECTION_MODE = "latest"`
- 如果你想回看某一轮，把 `ROUND_SELECTION_MODE` 改成 `manual`，并设置 `ROUND_INDEX_OVERRIDE`


In [ ]:
﻿# 这一节集中管理 notebook 的可配置项，尽量不在后续分析里写死路径。
import os


PROJECT_ROOT_OVERRIDE = None
EXPERIMENT_ROOT_OVERRIDE = None


def is_repo_root(candidate: Path | None) -> bool:
    if candidate is None:
        return False
    candidate = Path(candidate).resolve()
    return (candidate / "docs").exists() and (candidate / ".git").exists()


def iter_root_hints(start: Path) -> list[Path]:
    raw_hints = [
        PROJECT_ROOT_OVERRIDE,
        os.environ.get("ADL_PROJECT_ROOT"),
        os.environ.get("WORKSPACE_FOLDER"),
        os.environ.get("VSCODE_WORKSPACE_FOLDER"),
        os.environ.get("PWD"),
    ]

    direct_candidates = [start, *start.parents]
    seen: set[Path] = set()
    ordered: list[Path] = []

    def add_candidate(value) -> None:  # noqa: ANN001
        if value in (None, "", "None"):
            return
        candidate = Path(value).expanduser().resolve()
        if candidate not in seen:
            seen.add(candidate)
            ordered.append(candidate)

    for hint in raw_hints:
        add_candidate(hint)
    for candidate in direct_candidates:
        add_candidate(candidate)

    return ordered


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in iter_root_hints(start):
        if is_repo_root(candidate):
            return candidate
    return start


def looks_like_experiment_root(path: Path | None) -> bool:
    if path is None or not path.exists() or not path.is_dir():
        return False
    checks = [
        path / "configs" / "base.yaml",
        path / "data" / "processed",
        path / "models",
        path / "results",
    ]
    return any(item.exists() for item in checks)


def find_experiment_root(project_root: Path) -> Path:
    if EXPERIMENT_ROOT_OVERRIDE:
        override = Path(EXPERIMENT_ROOT_OVERRIDE).expanduser().resolve()
        if looks_like_experiment_root(override):
            return override

    env_override = os.environ.get("ADL_EXPERIMENT_ROOT")
    if env_override:
        env_path = Path(env_override).expanduser().resolve()
        if looks_like_experiment_root(env_path):
            return env_path

    candidates = [
        project_root / "minimal_adl_ethene_butadiene",
        project_root / "minimal_adl",
        project_root / "adl",
        project_root,
    ]
    for candidate in candidates:
        if looks_like_experiment_root(candidate):
            return candidate

    return project_root / "minimal_adl_ethene_butadiene"


def first_existing_path(paths: list[Path]) -> Path:
    for item in paths:
        if item.exists():
            return item
    return paths[0]


def build_standard_paths(experiment_root: Path) -> dict[str, Path]:
    candidates = {
        "results_dir": [
            experiment_root / "results",
            experiment_root / "output" / "results",
        ],
        "base_yaml": [
            experiment_root / "configs" / "base.yaml",
            experiment_root / "config" / "base.yaml",
        ],
        "delta_dataset_npz": [
            experiment_root / "data" / "processed" / "delta_dataset.npz",
            experiment_root / "results" / "delta_dataset.npz",
        ],
        "delta_dataset_metadata": [
            experiment_root / "data" / "processed" / "delta_dataset_metadata.json",
            experiment_root / "results" / "delta_dataset_metadata.json",
        ],
        "train_main_status": [
            experiment_root / "models" / "train_main_status.json",
            experiment_root / "results" / "train_main_status.json",
        ],
        "train_aux_status": [
            experiment_root / "models" / "train_aux_status.json",
            experiment_root / "results" / "train_aux_status.json",
        ],
        "training_summary": [
            experiment_root / "models" / "training_summary.json",
            experiment_root / "results" / "training_summary.json",
        ],
        "training_state": [
            experiment_root / "models" / "training_state.json",
            experiment_root / "results" / "training_state.json",
        ],
        "training_split": [
            experiment_root / "models" / "training_split.json",
            experiment_root / "results" / "training_split.json",
        ],
        "train_main_predictions": [
            experiment_root / "models" / "train_main_predictions.csv",
            experiment_root / "results" / "train_main_predictions.csv",
        ],
        "train_aux_predictions": [
            experiment_root / "models" / "train_aux_predictions.csv",
            experiment_root / "results" / "train_aux_predictions.csv",
        ],
        "train_main_history": [
            experiment_root / "models" / "train_main_history.json",
            experiment_root / "results" / "train_main_history.json",
        ],
        "train_aux_history": [
            experiment_root / "models" / "train_aux_history.json",
            experiment_root / "results" / "train_aux_history.json",
        ],
        "training_diagnostics": [
            experiment_root / "models" / "training_diagnostics.json",
            experiment_root / "results" / "training_diagnostics.json",
        ],
        "uncertainty_latest": [
            experiment_root / "results" / "uncertainty_latest.json",
            experiment_root / "models" / "uncertainty_latest.json",
        ],
        "active_learning_round_history": [
            experiment_root / "results" / "active_learning_round_history.json",
            experiment_root / "models" / "active_learning_round_history.json",
        ],
        "pipeline_run_summary": [
            experiment_root / "results" / "pipeline_run_summary.json",
            experiment_root / "pipeline_run_summary.json",
        ],
        "check_environment_report": [
            experiment_root / "results" / "check_environment_latest.json",
            experiment_root / "check_environment_latest.json",
        ],
    }
    return {key: first_existing_path(path_list) for key, path_list in candidates.items()}


PROJECT_ROOT = find_project_root()
ADL_ROOT = find_experiment_root(PROJECT_ROOT)

MODEL_VIEW_OPTIONS = {
    "main": {
        "history": ADL_ROOT / "models" / "train_main_history.json",
        "prediction": ADL_ROOT / "models" / "train_main_predictions.csv",
    },
    "aux": {
        "history": ADL_ROOT / "models" / "train_aux_history.json",
        "prediction": ADL_ROOT / "models" / "train_aux_predictions.csv",
    },
}

CONFIG = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "ADL_ROOT": ADL_ROOT,
    "MODEL_VIEW": "main",
    "TABLE_PRECISION": 4,
    "TABLE_MAX_ROWS": 30,
    "MAX_X_TICK_LABELS": 16,
    "FEATURE_TABLE_PATH": None,
    "HISTORY_PATH": None,
    "PREDICTION_PATH": None,
    "MODEL_PATH": None,
    "ID_COL": "sample_id",
    "TARGET_COL": None,
    "TRUE_COL": "y_true",
    "PRED_COL": "y_pred",
    "TASK_TYPE": "regression",
    "TOP_N_ERROR": 10,
    "TOP_N_UQ": 10,
    "TOP_N_MISSING": 12,
    "TOP_N_CATEGORY": 8,
    "TOP_N_IMPORTANCE": 15,
    "MAX_DESCRIPTOR_COLUMNS": 4,
    "MIN_CATEGORY_COUNT": 2,
    "MAX_PLOT_COLUMNS": 12,
    "MAX_HEATMAP_COLUMNS": 20,
    "HIGH_MISSING_THRESHOLD": 0.30,
    "HIGH_CORR_THRESHOLD": 0.90,
    "HIGH_SKEW_THRESHOLD": 1.0,
    "OUTLIER_RATIO_THRESHOLD": 0.05,
    "ROUND_SELECTION_MODE": "latest",
    "ROUND_INDEX_OVERRIDE": None,
    "RELATED_TO_TARGET_MIN_CORR": 0.03,
    "UQ_QUANTILE_BINS": 5,
    "MIN_UQ_MATCHED_SAMPLES": 20,
    "SMALL_VALIDATION_SIZE": 30,
}

model_view = CONFIG["MODEL_VIEW"]
if model_view in MODEL_VIEW_OPTIONS:
    CONFIG["HISTORY_PATH"] = CONFIG["HISTORY_PATH"] or MODEL_VIEW_OPTIONS[model_view]["history"]
    CONFIG["PREDICTION_PATH"] = CONFIG["PREDICTION_PATH"] or MODEL_VIEW_OPTIONS[model_view]["prediction"]

STANDARD_PATHS = build_standard_paths(ADL_ROOT)


def load_external_model(model_path: str | Path | None = None):
    model_path = model_path or CONFIG["MODEL_PATH"]
    if not model_path:
        return None

    model_path = Path(model_path)
    if not model_path.exists():
        print(f"未找到外部模型文件：{model_path}")
        return None

    suffix = model_path.suffix.lower()
    if suffix in {".pkl", ".pickle"}:
        with model_path.open("rb") as handle:
            return pickle.load(handle)

    if suffix == ".joblib":
        try:
            import joblib
        except Exception as exc:  # noqa: BLE001
            print(f"检测到 joblib 模型文件，但当前环境无法导入 joblib：{exc}")
            return None
        return joblib.load(model_path)

    print("当前默认加载器只支持 .pkl/.pickle/.joblib；如模型格式不同，请手动扩展 load_external_model()。")
    return None


print("项目根目录：", CONFIG["PROJECT_ROOT"])
print("实验目录：", CONFIG["ADL_ROOT"])
print("当前默认分析视角：", CONFIG["MODEL_VIEW"])
print("当前视觉预设：", THEME_PRESETS[ACTIVE_THEME_NAME]["display_name"])
print("可切换视觉预设：paper_blue_gray / presentation_high_contrast")
print("默认 history 文件：", CONFIG["HISTORY_PATH"])
print("默认 prediction 文件：", CONFIG["PREDICTION_PATH"])
print("轮次选择模式：", CONFIG["ROUND_SELECTION_MODE"])
print("提示：如果目录识别不正确，可设置 PROJECT_ROOT_OVERRIDE 或 EXPERIMENT_ROOT_OVERRIDE。")


## 1. 环境与文件检查

这一节的目标不是“立刻报错”，而是先把能读到什么、缺了什么说清楚。

我们会把文件分成三类：

- `仓库标准文件`：这套最小 ADL 流程默认会写出的文件
- `服务器扩展文件`：为了做更完整机器学习分析，你可以额外提供的文件
- `可选解释文件`：例如树模型文件、SHAP 相关输入，它们缺失时不阻塞主流程


In [ ]:
﻿# 这一节负责路径解析、文件读取、轮次识别和基础状态检查。
import re


ROUND_SELECTION_PATTERN = re.compile(r"round_(\d{3})_selection_summary\.json$")


def resolve_optional_path(path_like: Any) -> Path | None:
    if path_like in (None, "", "None"):
        return None
    path = Path(path_like)
    if path.is_absolute():
        return path
    return (CONFIG["PROJECT_ROOT"] / path).resolve()


def safe_read_json(path: Path | None, default: Any = None) -> Any:
    if path is None or not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as exc:  # noqa: BLE001
        print(f"读取 JSON 失败：{path} -> {exc}")
        return default


def safe_read_yaml(path: Path | None, default: Any = None) -> Any:
    if path is None or not path.exists():
        return default
    try:
        return yaml.safe_load(path.read_text(encoding="utf-8"))
    except Exception as exc:  # noqa: BLE001
        print(f"读取 YAML 失败：{path} -> {exc}")
        return default


def safe_read_npz(path: Path | None) -> dict[str, np.ndarray]:
    if path is None or not path.exists():
        return {}
    try:
        with np.load(path, allow_pickle=True) as payload:
            return {key: payload[key] for key in payload.files}
    except Exception as exc:  # noqa: BLE001
        print(f"读取 NPZ 失败：{path} -> {exc}")
        return {}


def safe_read_table(path: Path | None) -> pd.DataFrame:
    if path is None or not path.exists():
        return pd.DataFrame()

    suffix = path.suffix.lower()
    try:
        if suffix == ".csv":
            return pd.read_csv(path)
        if suffix == ".parquet":
            return pd.read_parquet(path)
        if suffix in {".json", ".jsonl"}:
            try:
                return pd.read_json(path)
            except ValueError:
                return pd.read_json(path, lines=True)
        if suffix in {".pkl", ".pickle"}:
            return pd.read_pickle(path)
        if suffix == ".npz":
            npz_payload = safe_read_npz(path)
            normalized = {}
            for key, value in npz_payload.items():
                arr = np.asarray(value)
                if arr.ndim == 1:
                    normalized[key] = arr
            return pd.DataFrame(normalized)
    except Exception as exc:  # noqa: BLE001
        print(f"读取表格失败：{path} -> {exc}")
        return pd.DataFrame()

    print(f"暂不支持自动读取该格式：{path.suffix}，请手动扩展 safe_read_table()。")
    return pd.DataFrame()


def safe_read_history(path: Path | None) -> dict[str, list[float]]:
    if path is None or not path.exists():
        return {}

    suffix = path.suffix.lower()
    try:
        if suffix in {".json", ".jsonl"}:
            payload = safe_read_json(path, default={})
            if isinstance(payload, dict) and isinstance(payload.get("history"), dict):
                payload = payload["history"]
            if isinstance(payload, dict):
                return {
                    str(key): [float(item) for item in np.ravel(value).tolist()]
                    for key, value in payload.items()
                    if isinstance(value, (list, tuple, np.ndarray))
                }

        if suffix == ".csv":
            frame = pd.read_csv(path)
            return {str(column): frame[column].dropna().tolist() for column in frame.columns}

        if suffix in {".pkl", ".pickle"}:
            payload = pd.read_pickle(path)
            if hasattr(payload, "history") and isinstance(payload.history, dict):
                payload = payload.history
            if isinstance(payload, dict):
                return {
                    str(key): [float(item) for item in np.ravel(value).tolist()]
                    for key, value in payload.items()
                }
    except Exception as exc:  # noqa: BLE001
        print(f"读取 history 失败：{path} -> {exc}")
        return {}

    print("history 文件已找到，但当前无法自动转换成逐 epoch 指标字典。")
    return {}


def normalize_sample_id_array(values: Any) -> list[str]:
    normalized = []
    for item in np.asarray(values).tolist():
        if isinstance(item, bytes):
            normalized.append(item.decode("utf-8"))
        else:
            normalized.append(str(item))
    return normalized


def module_available(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None


def parse_int_like(value: Any) -> int | None:
    if value in (None, "", "None"):
        return None
    try:
        return int(value)
    except Exception:
        return None


def parse_round_index(path_like: str | Path | None) -> int | None:
    if path_like is None:
        return None
    match = ROUND_SELECTION_PATTERN.search(Path(path_like).name)
    return int(match.group(1)) if match else None


def parse_round_from_directory(path_like: str | Path | None) -> int | None:
    if path_like is None:
        return None
    for item in Path(path_like).parts[::-1]:
        if item.lower().startswith("round_"):
            digits = "".join(ch for ch in item if ch.isdigit())
            if digits:
                return int(digits)
    return None


def safe_manifest_count(path_like: str | Path | None) -> int | None:
    path = Path(path_like) if path_like else None
    if path is None or not path.exists():
        return None
    payload = safe_read_json(path, default=[])
    return len(payload) if isinstance(payload, list) else None


def normalize_selected_ids(payload: dict[str, Any]) -> list[str]:
    selected_ids = payload.get("selected_sample_ids")
    if isinstance(selected_ids, list) and selected_ids:
        return [str(item) for item in selected_ids]

    selected_samples = payload.get("selected_samples", [])
    if isinstance(selected_samples, list):
        if selected_samples and isinstance(selected_samples[0], dict):
            return [str(item.get("sample_id")) for item in selected_samples if item.get("sample_id") is not None]
        return [str(item) for item in selected_samples]
    return []


def resolve_round_index_from_sources(payload: dict[str, Any], summary_path: Path, manifest_path: Path) -> tuple[int, str, dict[str, Any]]:
    payload_idx = parse_int_like(payload.get("round_index"))
    filename_idx = parse_round_index(summary_path)
    manifest_idx = parse_round_index(manifest_path)
    directory_idx = parse_round_from_directory(summary_path)

    candidates = [
        ("payload", payload_idx),
        ("filename", filename_idx),
        ("manifest", manifest_idx),
        ("directory", directory_idx),
    ]
    for source, value in candidates:
        if value is not None:
            resolved = int(value)
            break
    else:
        source, resolved = ("fallback", 0)

    details = {
        "round_index_payload": payload_idx,
        "round_index_filename": filename_idx,
        "round_index_manifest": manifest_idx,
        "round_index_directory": directory_idx,
        "round_index_consistent": len({v for _, v in candidates if v is not None}) <= 1,
    }
    return resolved, source, details


def build_round_history_row(summary_path: Path) -> dict[str, Any]:
    payload = safe_read_json(summary_path, default={}) or {}

    manifest_hint = parse_round_index(summary_path)
    if manifest_hint is None:
        manifest_path = summary_path.parent / "round_selected_manifest.json"
    else:
        manifest_path = summary_path.parent / f"round_{manifest_hint:03d}_selected_manifest.json"

    round_index, round_source, round_details = resolve_round_index_from_sources(payload, summary_path, manifest_path)
    selected_sample_ids = normalize_selected_ids(payload)
    selected_count = parse_int_like(payload.get("selected_count"))
    if selected_count is None:
        selected_count = parse_int_like(payload.get("num_selected"))
    if selected_count is None:
        selected_count = len(selected_sample_ids)

    manifest_count = safe_manifest_count(manifest_path)

    row = {
        "round_index": int(round_index),
        "round_index_source": round_source,
        "round_index_note": "一致" if round_details["round_index_consistent"] else "来源存在差异",
        "summary_selected_count": int(selected_count),
        "selected_count": int(selected_count),
        "manifest_selected_count": manifest_count,
        "selected_count_consistent": None if manifest_count is None else manifest_count == int(selected_count),
        "selected_manifest_exists": manifest_path.exists(),
        "selected_sample_ids": selected_sample_ids,
        "num_pool_samples": payload.get("num_pool_samples"),
        "num_uncertain_samples": payload.get("num_uncertain_samples"),
        "uncertain_ratio": payload.get("uncertain_ratio"),
        "converged": payload.get("converged"),
        "selection_summary_file": str(summary_path),
        "selection_manifest_file": str(manifest_path),
        "updated_at": payload.get("updated_at") or payload.get("created_at"),
    }
    row.update(round_details)
    return row


def build_round_history_row_from_payload(item: dict[str, Any], results_dir: Path) -> dict[str, Any] | None:
    if not isinstance(item, dict):
        return None

    round_index = parse_int_like(item.get("round_index"))
    if round_index is None:
        return None

    summary_file = item.get("selection_summary_file") or str(results_dir / f"round_{round_index:03d}_selection_summary.json")
    summary_path = Path(summary_file)

    manifest_file = item.get("selection_manifest_file") or str(results_dir / f"round_{round_index:03d}_selected_manifest.json")
    manifest_path = Path(manifest_file)

    selected_ids = item.get("selected_sample_ids", [])
    if not isinstance(selected_ids, list):
        selected_ids = []

    selected_count = parse_int_like(item.get("selected_count"))
    if selected_count is None:
        selected_count = len(selected_ids)

    manifest_count = parse_int_like(item.get("manifest_selected_count"))
    if manifest_count is None:
        manifest_count = safe_manifest_count(manifest_path)

    row = {
        "round_index": int(round_index),
        "round_index_source": item.get("round_index_source", "history_payload"),
        "round_index_note": item.get("round_index_note", "来自 active_learning_round_history.json"),
        "round_index_payload": parse_int_like(item.get("round_index_payload")),
        "round_index_filename": parse_int_like(item.get("round_index_filename")),
        "round_index_manifest": parse_int_like(item.get("round_index_manifest")),
        "round_index_directory": parse_int_like(item.get("round_index_directory")),
        "round_index_consistent": item.get("round_index_consistent", True),
        "selected_count": int(selected_count),
        "summary_selected_count": int(selected_count),
        "manifest_selected_count": manifest_count,
        "selected_count_consistent": item.get("selected_count_consistent"),
        "selected_manifest_exists": bool(item.get("selected_manifest_exists", manifest_path.exists())),
        "selected_sample_ids": [str(v) for v in selected_ids],
        "num_pool_samples": item.get("num_pool_samples"),
        "num_uncertain_samples": item.get("num_uncertain_samples"),
        "uncertain_ratio": item.get("uncertain_ratio"),
        "converged": item.get("converged"),
        "selection_summary_file": str(summary_path),
        "selection_manifest_file": str(manifest_path),
        "updated_at": item.get("updated_at") or item.get("created_at"),
    }

    if row["selected_count_consistent"] is None and manifest_count is not None:
        row["selected_count_consistent"] = manifest_count == int(selected_count)

    return row


def build_round_history_df(results_dir: Path, history_payload: dict[str, Any]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    if isinstance(history_payload, dict):
        round_rows = history_payload.get("rounds", [])
        if isinstance(round_rows, list):
            for item in round_rows:
                row = build_round_history_row_from_payload(item, results_dir)
                if row is not None:
                    rows.append(row)

    if results_dir.exists():
        for summary_path in sorted(results_dir.glob("round_*_selection_summary.json")):
            rows.append(build_round_history_row(summary_path))

    if not rows:
        return pd.DataFrame()

    frame = pd.DataFrame(rows)

    file_score = frame.get("selection_summary_file", pd.Series([""] * len(frame))).astype(str).map(lambda item: int(Path(item).exists()) if item else 0)
    consistency_score = frame.get("round_index_consistent", pd.Series([False] * len(frame))).fillna(False).astype(int)
    frame["_merge_quality"] = file_score * 3 + consistency_score

    frame = (
        frame.sort_values(["round_index", "_merge_quality"], ascending=[True, False])
        .drop_duplicates(subset=["round_index"], keep="first")
        .drop(columns=["_merge_quality"])
        .reset_index(drop=True)
    )

    if "selected_sample_ids" not in frame.columns:
        frame["selected_sample_ids"] = [[] for _ in range(len(frame))]

    frame["selected_count"] = pd.to_numeric(frame["selected_count"], errors="coerce").fillna(0).astype(int)
    frame["selected_manifest_exists"] = frame.get("selected_manifest_exists", pd.Series([False] * len(frame))).fillna(False).astype(bool)
    frame["round_index"] = pd.to_numeric(frame["round_index"], errors="coerce").fillna(0).astype(int)

    frame = frame.sort_values("round_index").reset_index(drop=True)
    frame["round_sequence"] = np.arange(1, len(frame) + 1)
    frame["round_plot_label"] = frame["round_index"].map(lambda idx: f"R{int(idx):03d}")

    pool_size = base_config.get("sampling", {}).get("default_pool_size") if isinstance(base_config, dict) else None
    initial_points = base_config.get("active_learning", {}).get("initial_points") if isinstance(base_config, dict) else None
    cumulative_selected_before = frame["selected_count"].shift(fill_value=0).cumsum()

    if initial_points is not None:
        frame["labeled_samples_before_selection"] = int(initial_points) + cumulative_selected_before
        frame["expected_labeled_after_selection"] = frame["labeled_samples_before_selection"] + frame["selected_count"]
    else:
        frame["labeled_samples_before_selection"] = np.nan
        frame["expected_labeled_after_selection"] = np.nan

    if pool_size is not None:
        frame["remaining_candidate_samples"] = int(pool_size) - frame["labeled_samples_before_selection"]
    else:
        frame["remaining_candidate_samples"] = np.nan

    if "selected_count_consistent" in frame.columns:
        frame["selected_count_consistent"] = frame["selected_count_consistent"].where(frame["selected_manifest_exists"], np.nan)

    return frame


def build_round_index_diagnostics(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    if frame.empty:
        return pd.DataFrame(), []

    diagnostics_cols = [
        "round_index",
        "round_index_source",
        "round_index_payload",
        "round_index_filename",
        "round_index_manifest",
        "round_index_directory",
        "round_index_consistent",
        "round_index_note",
        "selection_summary_file",
    ]
    diagnostics = frame[[col for col in diagnostics_cols if col in frame.columns]].copy()

    warnings_list = []
    inconsistent = diagnostics.loc[~diagnostics.get("round_index_consistent", pd.Series([True] * len(diagnostics))).fillna(True)]
    for _, row in inconsistent.head(20).iterrows():
        warnings_list.append(
            f"第 {int(row['round_index'])} 轮存在来源不一致：payload={row.get('round_index_payload')}，"
            f"文件名={row.get('round_index_filename')}，manifest={row.get('round_index_manifest')}。"
        )

    sorted_rounds = sorted(frame["round_index"].dropna().astype(int).tolist())
    for left, right in zip(sorted_rounds[:-1], sorted_rounds[1:]):
        if right - left > 1:
            warnings_list.append(f"轮次序列存在缺口：第 {left} 轮之后直接到第 {right} 轮。")

    return diagnostics, warnings_list


def pick_current_round(frame: pd.DataFrame) -> tuple[int | None, dict[str, Any], pd.Series | None]:
    if frame.empty:
        return None, {}, None

    mode = CONFIG.get("ROUND_SELECTION_MODE", "latest")
    override = CONFIG.get("ROUND_INDEX_OVERRIDE")
    current_row = None
    if mode == "manual" and override is not None:
        matched = frame.loc[frame["round_index"] == int(override)]
        if not matched.empty:
            current_row = matched.iloc[-1]

    if current_row is None:
        current_row = frame.sort_values("round_index").iloc[-1]

    summary_path = Path(current_row.get("selection_summary_file")) if current_row.get("selection_summary_file") else None
    current_payload = safe_read_json(summary_path, default={}) if summary_path else {}
    current_payload = current_payload if isinstance(current_payload, dict) else {}
    current_payload.setdefault("round_index", int(current_row["round_index"]))
    current_payload.setdefault("selected_sample_ids", current_row.get("selected_sample_ids", []))
    current_payload.setdefault("selected_count", int(current_row.get("selected_count", 0)))

    return int(current_row["round_index"]), current_payload, current_row


OPTIONAL_PATHS = {
    "feature_table": resolve_optional_path(CONFIG["FEATURE_TABLE_PATH"]),
    "history": resolve_optional_path(CONFIG["HISTORY_PATH"]),
    "prediction": resolve_optional_path(CONFIG["PREDICTION_PATH"]),
    "model": resolve_optional_path(CONFIG["MODEL_PATH"]),
}

base_config = safe_read_yaml(STANDARD_PATHS["base_yaml"], default={}) or {}
delta_dataset = safe_read_npz(STANDARD_PATHS["delta_dataset_npz"])
delta_metadata = safe_read_json(STANDARD_PATHS["delta_dataset_metadata"], default={}) or {}
train_main_status = safe_read_json(STANDARD_PATHS["train_main_status"], default={}) or {}
train_aux_status = safe_read_json(STANDARD_PATHS["train_aux_status"], default={}) or {}
training_summary = safe_read_json(STANDARD_PATHS["training_summary"], default={}) or {}
training_state = safe_read_json(STANDARD_PATHS["training_state"], default={}) or {}
training_split = safe_read_json(STANDARD_PATHS["training_split"], default={}) or {}
training_diagnostics = safe_read_json(STANDARD_PATHS["training_diagnostics"], default={}) or {}
pipeline_summary = safe_read_json(STANDARD_PATHS["pipeline_run_summary"], default={}) or {}
uncertainty_payload = safe_read_json(STANDARD_PATHS["uncertainty_latest"], default={}) or {}
round_history_payload = safe_read_json(STANDARD_PATHS["active_learning_round_history"], default={}) or {}

external_feature_df = safe_read_table(OPTIONAL_PATHS["feature_table"])
external_prediction_df = safe_read_table(OPTIONAL_PATHS["prediction"])
history_raw_payload = safe_read_json(OPTIONAL_PATHS["history"], default={}) or {}
history_payload = safe_read_history(OPTIONAL_PATHS["history"])

round_history_df = build_round_history_df(STANDARD_PATHS["results_dir"], round_history_payload)
round_index_diagnostics_df, round_index_warnings = build_round_index_diagnostics(round_history_df)
latest_round_index = None if round_history_df.empty else int(round_history_df["round_index"].max())
total_rounds = int(len(round_history_df))
current_round_index, current_selection_summary, current_round_row = pick_current_round(round_history_df)
selection_summary = current_selection_summary

file_rows = []
for name, path in STANDARD_PATHS.items():
    if name == "results_dir":
        continue
    file_rows.append(
        {
            "类别": "仓库标准文件",
            "文件键": name,
            "路径": str(path),
            "是否存在": path.exists(),
            "是否必需": name in {
                "base_yaml",
                "delta_dataset_metadata",
                "training_summary",
                "training_diagnostics",
                "uncertainty_latest",
            },
        }
    )

for name, path in OPTIONAL_PATHS.items():
    file_rows.append(
        {
            "类别": "可配置分析输入",
            "文件键": name,
            "路径": "" if path is None else str(path),
            "是否存在": False if path is None else path.exists(),
            "是否必需": False,
        }
    )

file_status_df = pd.DataFrame(file_rows)
display_table(file_status_df, title="环境与文件检查概览", rename_columns=False, precision=3, gradient_subset=[])

print("可选依赖检查：")
print(f"- seaborn 是否可用：{HAS_SEABORN}")
print(f"- shap 是否可用：{module_available('shap')}")
print(f"- scikit-learn 是否可用：{module_available('sklearn')}")

if isinstance(training_diagnostics, dict) and training_diagnostics:
    print("检测到 training_diagnostics.json，说明训练阶段已导出 notebook 所需分析入口。")
if isinstance(pipeline_summary, dict) and pipeline_summary:
    print(f"检测到 pipeline_run_summary.json，最近一次主控流程 success={pipeline_summary.get('success')}")

if round_history_df.empty:
    print("当前还没有检测到 round_*_selection_summary.json，因此多轮历史暂时为空。")
else:
    print(f"已识别到 {total_rounds} 轮主动学习选点记录，最新轮次是 round_{latest_round_index:03d}。")
    if current_round_index is not None:
        print(f"当前 notebook 默认分析的轮次是 round_{current_round_index:03d}。")

if round_index_warnings:
    print("轮次编号一致性提示：")
    for item in round_index_warnings[:8]:
        print(f"- {item}")

if not file_status_df["是否存在"].all():
    missing_df = file_status_df.loc[~file_status_df["是否存在"], ["类别", "文件键", "路径", "是否必需"]]
    display_table(missing_df, title="当前缺失的文件", rename_columns=False, precision=3, gradient_subset=[])
    print("提示：缺失文件不会让整本 notebook 直接崩掉；后续对应模块会给出跳过说明。")

print("这一节说明什么：先确认当前到底能分析哪些内容，再统一轮次来源，避免把路径问题误判成模型问题。")


## 2. 数据长什么样？

这一节重点回答：

- 数据规模有多大
- 每一列分别是什么类型
- 有没有缺失值、异常值、偏态和高度相关
- 这些问题会不会影响后续建模

如果服务器上已经准备好了外部特征表，就优先用外部特征表做 EDA；否则就从 ADL 的 `delta_dataset.npz` 和 metadata 中构造一个可分析 DataFrame。


In [ ]:
# 这一个代码块负责构造分析 DataFrame，并完成完整的 EDA。
def flatten_metadata_field(payload: Any, prefix: str = "meta") -> dict[str, Any]:
    # 把 metadata 中的嵌套字典尽量展开，方便后续做表格分析。
    if not isinstance(payload, dict):
        return {}
    flattened = {}
    for key, value in payload.items():
        flat_key = f"{prefix}_{key}"
        if isinstance(value, dict):
            for inner_key, inner_value in value.items():
                flattened[f"{flat_key}_{inner_key}"] = inner_value
        else:
            flattened[flat_key] = value
    return flattened


def build_adl_analysis_df(dataset_payload: dict[str, np.ndarray], metadata_payload: dict[str, Any]) -> pd.DataFrame:
    # 根据仓库标准输出构造一个更适合数据分析的样本级表格。
    if not dataset_payload:
        return pd.DataFrame()

    sample_ids = normalize_sample_id_array(dataset_payload.get("sample_ids", []))
    total_samples = len(sample_ids)
    if total_samples == 0:
        return pd.DataFrame()

    records = []
    sample_metadata_list = metadata_payload.get("samples", []) if isinstance(metadata_payload, dict) else []
    metadata_by_id = {
        str(item.get("sample_id")): item
        for item in sample_metadata_list
        if isinstance(item, dict) and item.get("sample_id") is not None
    }

    atomic_numbers = np.asarray(dataset_payload.get("atomic_numbers", []), dtype=object)
    coordinates = np.asarray(dataset_payload.get("coordinates", []), dtype=object)
    e_baseline = np.asarray(dataset_payload.get("E_baseline", np.full(total_samples, np.nan)), dtype=float)
    e_target = np.asarray(dataset_payload.get("E_target", np.full(total_samples, np.nan)), dtype=float)
    delta_e = np.asarray(dataset_payload.get("delta_E", np.full(total_samples, np.nan)), dtype=float)
    f_baseline = np.asarray(dataset_payload.get("F_baseline", np.zeros((total_samples, 1, 3))), dtype=float)
    f_target = np.asarray(dataset_payload.get("F_target", np.zeros((total_samples, 1, 3))), dtype=float)
    delta_f = np.asarray(dataset_payload.get("delta_F", np.zeros((total_samples, 1, 3))), dtype=float)

    for idx, sample_id in enumerate(sample_ids):
        sample_meta = metadata_by_id.get(sample_id, {})
        atomic_number_array = np.asarray(atomic_numbers[idx]) if idx < len(atomic_numbers) else np.asarray([])
        coordinate_array = np.asarray(coordinates[idx]) if idx < len(coordinates) else np.asarray([])
        record = {
            CONFIG["ID_COL"]: sample_id,
            "E_baseline": e_baseline[idx] if idx < len(e_baseline) else np.nan,
            "E_target": e_target[idx] if idx < len(e_target) else np.nan,
            "delta_E": delta_e[idx] if idx < len(delta_e) else np.nan,
            "|F_baseline|": float(np.linalg.norm(np.ravel(f_baseline[idx]))) if idx < len(f_baseline) else np.nan,
            "|F_target|": float(np.linalg.norm(np.ravel(f_target[idx]))) if idx < len(f_target) else np.nan,
            "|delta_F|": float(np.linalg.norm(np.ravel(delta_f[idx]))) if idx < len(delta_f) else np.nan,
            "num_atoms": int(len(atomic_number_array)),
            "charge": sample_meta.get("charge"),
            "multiplicity": sample_meta.get("multiplicity"),
            "source": sample_meta.get("source", ""),
            "geometry_file": sample_meta.get("geometry_file", ""),
            "mean_atomic_number": float(np.mean(atomic_number_array)) if atomic_number_array.size else np.nan,
            "coordinate_std": float(np.std(coordinate_array)) if coordinate_array.size else np.nan,
        }
        record.update(flatten_metadata_field(sample_meta.get("manifest_metadata", {}), prefix="manifest"))
        records.append(record)

    return pd.DataFrame(records)


def merge_external_and_adl(feature_df: pd.DataFrame, adl_df: pd.DataFrame) -> tuple[pd.DataFrame, str]:
    # 如果同时存在外部特征表和 ADL 标准表，则优先用外部表，并按 sample_id 尽量补齐 ADL 字段。
    if feature_df.empty and adl_df.empty:
        return pd.DataFrame(), "未检测到可分析的数据表。"
    if feature_df.empty:
        return adl_df.copy(), "当前使用 ADL 标准结果构造的数据表。"
    if adl_df.empty:
        return feature_df.copy(), "当前使用服务器提供的外部特征表。"

    left = feature_df.copy()
    right = adl_df.copy()
    if CONFIG["ID_COL"] in left.columns and CONFIG["ID_COL"] in right.columns:
        merged = left.merge(right, on=CONFIG["ID_COL"], how="left", suffixes=("", "_adl"))
        return merged, "当前以服务器外部特征表为主，并补充了 ADL 标准字段。"
    return left, "检测到外部特征表，但没有可用于合并的 sample_id，因此仅分析外部特征表。"


def detect_column_groups(frame: pd.DataFrame) -> tuple[list[str], list[str]]:
    # 自动区分数值列和类别列，便于分别处理。
    if frame.empty:
        return [], []
    numeric_cols = frame.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [column for column in frame.columns if column not in numeric_cols]
    return numeric_cols, categorical_cols


def infer_target_column(frame: pd.DataFrame) -> str | None:
    # 优先使用用户显式设置的 TARGET_COL；没有时尝试按 ADL 默认目标 delta_E 推断。
    if CONFIG["TARGET_COL"] and CONFIG["TARGET_COL"] in frame.columns:
        return CONFIG["TARGET_COL"]
    if "delta_E" in frame.columns:
        return "delta_E"
    if CONFIG["TRUE_COL"] in frame.columns:
        return CONFIG["TRUE_COL"]
    return None


def summarize_outliers(frame: pd.DataFrame, numeric_cols: list[str]) -> pd.DataFrame:
    # 用 IQR 粗略统计每个数值特征的异常值比例。
    rows = []
    for column in numeric_cols:
        series = frame[column].dropna()
        if series.empty:
            continue
        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            outlier_ratio = 0.0
        else:
            lower = q1 - 1.5 * iqr
            upper = q3 + 1.5 * iqr
            outlier_ratio = ((series < lower) | (series > upper)).mean()
        rows.append({"特征": column, "异常值比例": outlier_ratio})
    return pd.DataFrame(rows).sort_values("异常值比例", ascending=False)



def plot_missingness(frame: pd.DataFrame) -> None:
    # 只展示真正存在缺失的字段。
    if frame.empty:
        print("当前分析表为空，无法绘制缺失率图。")
        return

    missing_rate = frame.isna().mean().sort_values(ascending=False)
    nonzero_missing = missing_rate[missing_rate > 0]
    zero_missing_count = int((missing_rate == 0).sum())
    if nonzero_missing.empty:
        print("所有字段缺失率都为 0，跳过缺失率图。")
        return

    top_n = min(get_config_value("TOP_N_MISSING", 12), len(nonzero_missing))
    top_missing = nonzero_missing.head(top_n).sort_values(ascending=True)
    fig, ax = create_barh_figure(len(top_missing), width=10.8, base=3.8, step=0.42, max_height=9.8)
    ax.barh(
        clean_long_labels(top_missing.index, width=14, max_chars=30),
        top_missing.values,
        color=build_palette(len(top_missing)),
        edgecolor="white",
        linewidth=0.8,
    )
    style_axis(ax, "缺失率最高的字段", "缺失率", "字段名称", grid_axis="x")
    format_percent_axis(ax, axis="x", decimals=1 if float(top_missing.max()) < 0.10 else 0)
    expand_axis_limit_for_labels(ax, float(top_missing.max()), orientation="horizontal", margin_ratio=0.18)
    annotate_bars(ax, orientation="horizontal", as_percent=True, decimals=1, skip_zero=True)
    finalize_plot(fig)
    print("这张图说明什么：它只保留缺失率大于 0 的字段，帮助我们优先关注真正需要处理的字段。")
    if zero_missing_count:
        print(f"已过滤掉 {zero_missing_count} 个缺失率为 0 的字段。")
    if len(nonzero_missing) > top_n:
        print(f"共有 {len(nonzero_missing)} 个字段存在缺失，这里默认仅展示前 {top_n} 个。")

    heatmap_frame = frame[nonzero_missing.head(top_n).index].isna().astype(int).copy()
    if heatmap_frame.empty or heatmap_frame.shape[1] == 0:
        print("缺失模式热图没有可展示的字段，跳过热图。")
        return
    heatmap_frame.columns = clean_long_labels(heatmap_frame.columns, width=10, max_chars=24)
    fig, ax = create_heatmap_figure(
        len(heatmap_frame),
        len(heatmap_frame.columns),
        min_width=10.8,
        max_width=15.0,
        min_height=4.8,
        max_height=10.8,
        width_per_col=0.42,
        height_per_row=0.022,
    )
    sns.heatmap(
        heatmap_frame,
        cbar=True,
        yticklabels=False,
        cmap=REPORT_HEATMAP_CMAP,
        linewidths=0.0,
        cbar_kws={"label": "是否缺失"},
        ax=ax,
    )
    if ax.collections:
        style_colorbar(ax.collections[0].colorbar, "是否缺失")
    style_axis(ax, "缺失模式热图", "字段名称", "样本", grid_axis="none")
    clean_long_labels(ax, axis="x", rotation=0, width=10, ha="center", max_chars=24)
    finalize_plot(fig)
    print("这张图说明什么：它只显示存在缺失的字段，帮助观察缺失是否集中在同一批样本上。")


def filter_columns_by_target_relevance(frame: pd.DataFrame, columns: list[str], target_col: str | None) -> tuple[list[str], list[str]]:
    # ?????????????????????????????????
    if target_col is None or target_col not in frame.columns:
        return columns, []

    target = pd.to_numeric(frame[target_col], errors="coerce")
    relevant: list[str] = []
    skipped: list[str] = []
    corr_threshold = float(CONFIG.get("RELATED_TO_TARGET_MIN_CORR", 0.03))

    for column in columns:
        if column == target_col or column not in frame.columns:
            continue
        col_series = pd.to_numeric(frame[column], errors="coerce")
        paired = pd.DataFrame({"x": col_series, "y": target}).dropna()
        if len(paired) < 8 or paired["x"].nunique() <= 1 or paired["y"].nunique() <= 1:
            skipped.append(column)
            continue
        corr = paired["x"].corr(paired["y"])
        if pd.isna(corr) or abs(float(corr)) < corr_threshold:
            skipped.append(column)
        else:
            relevant.append(column)

    return (relevant if relevant else columns), skipped


def plot_numeric_histograms(frame: pd.DataFrame, numeric_cols: list[str]) -> None:
    # ??????????????????????????????
    selected_cols, skipped_cols = select_informative_numeric_columns(frame, numeric_cols, max_cols=CONFIG["MAX_PLOT_COLUMNS"])
    target_col = infer_target_column(frame)
    selected_cols, skipped_by_relevance = filter_columns_by_target_relevance(frame, selected_cols, target_col)
    skipped_cols.extend(skipped_by_relevance)

    if not selected_cols:
        print("?????????????????????")
        return

    n_cols = 2 if len(selected_cols) <= 4 else 3
    n_rows = math.ceil(len(selected_cols) / n_cols)
    fig, axes = create_subplots(
        n_rows,
        n_cols,
        kind="wide",
        width_scale=0.96,
        height_scale=max(1.0, n_rows * 0.88),
    )
    axes = np.array(axes).reshape(-1)
    for idx, (ax, column) in enumerate(zip(axes, selected_cols)):
        series = pd.to_numeric(frame[column], errors="coerce").dropna()
        color = REPORT_PALETTE[idx % len(REPORT_PALETTE)]
        sns.histplot(series, kde=True, ax=ax, color=color, bins=min(24, max(12, int(np.sqrt(len(series) or 1)))))
        ax.axvline(series.median(), color=REPORT_COLORS["orange"], linestyle="--", linewidth=1.2, alpha=0.85)
        style_axis(ax, "{}??".format(clean_long_labels(column, width=12, max_chars=20)), to_cn_label(column), "???", grid_axis="y")
        set_integer_ticks(ax, axis="y")
    for ax in axes[len(selected_cols):]:
        ax.axis("off")
    finalize_plot(fig, title="??????")
    print("????????????????????????????????????????????????")
    note = build_filtered_note(skipped_cols, label="????")
    if note:
        print(note)

def plot_boxplots(frame: pd.DataFrame, numeric_cols: list[str]) -> None:
    # 箱线图只保留真正有分布差异的字段。
    selected_cols, skipped_cols = select_informative_numeric_columns(frame, numeric_cols, max_cols=CONFIG["MAX_PLOT_COLUMNS"])
    if not selected_cols:
        print("当前没有适合绘制箱线图的高信息量数值字段。")
        return

    display_frame = frame[selected_cols].copy()
    display_frame.columns = [to_cn_label(column) for column in display_frame.columns]
    fig, ax = create_barh_figure(len(selected_cols), width=11.2, base=4.0, step=0.34, max_height=11.2)
    sns.boxplot(
        data=display_frame,
        orient="h",
        ax=ax,
        color="#8FAFCA",
        width=0.62,
        fliersize=3,
        linewidth=1.1,
    )
    style_axis(ax, "数值字段箱线图", "数值范围", "字段名称", grid_axis="x")
    clean_long_labels(ax, axis="y", width=16, max_chars=28)
    finalize_plot(fig)
    print("这张图说明什么：它只保留有实际波动的数值字段，帮助快速识别偏态和异常值。")
    note = build_filtered_note(skipped_cols, label="数值字段")
    if note:
        print(note)


def plot_correlation_heatmap(frame: pd.DataFrame, numeric_cols: list[str]) -> None:
    # 相关性热图只保留有实际变化的字段。
    selected_cols, skipped_cols = select_informative_numeric_columns(frame, numeric_cols, max_cols=CONFIG["MAX_HEATMAP_COLUMNS"])
    if len(selected_cols) < 2:
        print("高信息量数值字段少于 2 个，跳过相关性热图。")
        return

    corr = frame[selected_cols].corr(numeric_only=True)
    corr = corr.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if corr.empty or corr.shape[0] < 2 or corr.shape[1] < 2:
        print("相关性矩阵没有足够可展示的有效字段，跳过相关性热图。")
        return

    corr.index = [to_cn_label(column) for column in corr.index]
    corr.columns = [to_cn_label(column) for column in corr.columns]
    fig, ax = create_heatmap_figure(
        len(corr.index),
        len(corr.columns),
        min_width=8.8,
        max_width=14.8,
        min_height=6.8,
        max_height=11.2,
        width_per_col=0.32,
        height_per_row=0.18,
    )
    sns.heatmap(
        corr,
        cmap=REPORT_DIVERGING_CMAP,
        center=0,
        annot=len(corr.columns) <= 10,
        fmt=".2f",
        linewidths=0.5,
        square=False,
        vmin=-1,
        vmax=1,
        cbar_kws={"label": "相关系数"},
        ax=ax,
    )
    if ax.collections:
        style_colorbar(ax.collections[0].colorbar, "相关系数")
    style_axis(ax, "数值字段相关性热图", "字段名称", "字段名称", grid_axis="none")
    clean_long_labels(ax, axis="x", rotation=32, width=10, ha="right", max_chars=24)
    clean_long_labels(ax, axis="y", width=10, max_chars=24)
    finalize_plot(fig)
    print("这张图说明什么：它只保留有变化的数值字段，帮助定位真正值得关注的强相关关系。")
    note = build_filtered_note(skipped_cols, label="数值字段")
    if note:
        print(note)


def plot_categorical_distributions(frame: pd.DataFrame, categorical_cols: list[str]) -> None:
    # 类别分布图只保留有区分度的类别字段，并默认截断长尾。
    if not categorical_cols:
        print("当前没有可用于展示的类别字段。")
        return

    informative_cols = [column for column in categorical_cols if column in frame.columns and is_informative_categorical_series(frame[column])]
    skipped_cols = [column for column in categorical_cols if column not in informative_cols]
    if not informative_cols:
        print("当前没有适合展示的高信息量类别字段。")
        return

    selected_cols = informative_cols[: min(4, len(informative_cols))]
    skipped_cols.extend(informative_cols[len(selected_cols):])
    fig, axes = create_subplots(
        len(selected_cols),
        1,
        kind="wide",
        width_scale=0.98,
        height_scale=max(1.0, len(selected_cols) * 0.82),
    )
    axes = np.array(axes).reshape(-1)
    plotted_count = 0
    truncated_notes: list[str] = []
    for ax, column in zip(axes, selected_cols):
        top_n = get_config_value("TOP_N_CATEGORY", 8)
        counts, note = build_category_counts(frame[column], top_n=top_n, min_count=get_config_value("MIN_CATEGORY_COUNT", 2))
        if counts.empty:
            ax.axis("off")
            skipped_cols.append(column)
            continue
        plotted_count += 1
        if note:
            truncated_notes.append(f"{to_cn_label(column)}：{note}")
        ax.barh(
            clean_long_labels(counts.index, width=16, max_chars=30),
            counts.values,
            color=build_palette(len(counts)),
            edgecolor="white",
            linewidth=0.8,
        )
        title_suffix = f"（Top {top_n}）" if len(counts) >= top_n else ""
        style_axis(
            ax,
            f"{clean_long_labels(column, width=12, max_chars=22)} 类别分布{title_suffix}",
            "样本数",
            to_cn_label(column),
            grid_axis="x",
        )
        set_integer_ticks(ax, axis="x")
        expand_axis_limit_for_labels(ax, float(counts.max()), orientation="horizontal", margin_ratio=0.18)
        annotate_bars(ax, orientation="horizontal", decimals=0)
    for ax in axes[plotted_count:]:
        ax.axis("off")
    if plotted_count == 0:
        plt.close(fig)
        print("当前类别字段要么只有单一类别，要么全是低频长尾，跳过类别分布图。")
        return
    finalize_plot(fig, title="类别变量分布概览")
    print("这张图说明什么：它只展示有区分度的类别字段，并默认裁掉长尾类别，减少视觉噪音。")
    note = build_filtered_note(skipped_cols, label="类别字段")
    if note:
        print(note)
    for item in truncated_notes:
        print(item)


def plot_target_distribution(frame: pd.DataFrame, target_col: str | None) -> None:
    # 目标分布图只在目标列确实存在波动时才展示。
    if target_col is None or target_col not in frame.columns:
        print("当前没有可用于展示的目标列。")
        return

    series = pd.to_numeric(frame[target_col], errors="coerce").dropna()
    if series.empty:
        print("目标列没有有效数值，跳过目标分布图。")
        return
    if series.nunique(dropna=True) <= 1:
        print("目标列几乎没有变化，跳过目标分布图。")
        return

    fig, ax = create_figure("wide")
    sns.histplot(series, kde=True, color=REPORT_COLORS["rose"], ax=ax, bins=min(28, max(12, int(np.sqrt(len(series) or 1)))))
    ax.axvline(series.mean(), color=REPORT_COLORS["navy"], linestyle="--", linewidth=1.4, label="均值")
    ax.axvline(series.median(), color=REPORT_COLORS["orange"], linestyle=":", linewidth=1.6, label="中位数")
    style_axis(ax, "{}分布".format(to_cn_label(target_col)), to_cn_label(target_col), "样本数", grid_axis="y")
    set_integer_ticks(ax, axis="y")
    ax.legend(loc="upper right")
    finalize_plot(fig)
    print("这张图说明什么：它用于看目标值分布是否偏态、是否存在长尾以及均值和中位数差异。")


def summarize_eda_issues(frame: pd.DataFrame, numeric_cols: list[str], target_col: str | None) -> dict[str, Any]:
    # 自动汇总 EDA 中最值得注意的问题，并给出建模前预处理建议。
    summary = {
        "high_missing_cols": [],
        "high_skew_cols": [],
        "high_corr_pairs": [],
        "high_outlier_cols": [],
        "preprocess_suggestions": [],
    }
    if frame.empty:
        return summary

    missing_rate = frame.isna().mean()
    summary["high_missing_cols"] = missing_rate[missing_rate >= CONFIG["HIGH_MISSING_THRESHOLD"]].index.tolist()

    if numeric_cols:
        skew_series = frame[numeric_cols].skew(numeric_only=True).sort_values(key=np.abs, ascending=False)
        summary["high_skew_cols"] = skew_series[skew_series.abs() >= CONFIG["HIGH_SKEW_THRESHOLD"]].index.tolist()

        corr = frame[numeric_cols].corr(numeric_only=True).abs()
        for i, left_col in enumerate(corr.columns):
            for right_col in corr.columns[i + 1:]:
                value = corr.loc[left_col, right_col]
                if pd.notna(value) and value >= CONFIG["HIGH_CORR_THRESHOLD"]:
                    summary["high_corr_pairs"].append((left_col, right_col, float(value)))

        outlier_df = summarize_outliers(frame, numeric_cols)
        if not outlier_df.empty:
            summary["high_outlier_cols"] = outlier_df.loc[
                outlier_df["异常值比例"] >= CONFIG["OUTLIER_RATIO_THRESHOLD"], "特征"
            ].tolist()

    if summary["high_missing_cols"]:
        summary["preprocess_suggestions"].append("对缺失严重的字段先判断是否保留；必要时删列或采用分组补值。")
    if summary["high_skew_cols"]:
        summary["preprocess_suggestions"].append("对偏态明显的连续变量考虑对数变换、Box-Cox 或更稳健的缩放。")
    if summary["high_corr_pairs"]:
        summary["preprocess_suggestions"].append("对高度相关的特征考虑删减、合并或在解释时成组看待。")
    if summary["high_outlier_cols"]:
        summary["preprocess_suggestions"].append("对异常值比例高的字段考虑截尾、稳健模型或单独检查上游标注。")
    if target_col is not None and target_col in frame.columns and frame[target_col].dropna().skew() > CONFIG["HIGH_SKEW_THRESHOLD"]:
        summary["preprocess_suggestions"].append("目标变量偏态较强，可考虑对目标做变换，或在报告中解释高值样本更难拟合。")
    if not summary["preprocess_suggestions"]:
        summary["preprocess_suggestions"].append("当前未发现特别突出的数据问题，可以先用标准化 + 常规回归流程作为基线。")
    return summary


adl_analysis_df = build_adl_analysis_df(delta_dataset, delta_metadata)
adl_analysis_df = build_adl_analysis_df(delta_dataset, delta_metadata)
analysis_df, data_source_note = merge_external_and_adl(external_feature_df, adl_analysis_df)
numeric_columns, categorical_columns = detect_column_groups(analysis_df)
effective_target_col = infer_target_column(analysis_df)

print(data_source_note)
if analysis_df.empty:
    print("当前没有可分析的数据集。请在服务器环境中准备 FEATURE_TABLE_PATH，或确保 ADL 标准结果文件存在。")
    eda_summary = {
        "high_missing_cols": [],
        "high_skew_cols": [],
        "high_corr_pairs": [],
        "high_outlier_cols": [],
        "preprocess_suggestions": ["请先准备数据文件，再运行 EDA。"],
    }
else:
    pool_size = base_config.get("sampling", {}).get("default_pool_size") if isinstance(base_config, dict) else None
    current_labeled_samples = delta_metadata.get("num_samples") if isinstance(delta_metadata, dict) else None
    if current_labeled_samples is None:
        current_labeled_samples = analysis_df.shape[0]
    remaining_candidate_samples = (
        int(pool_size) - int(current_labeled_samples)
        if pool_size is not None and current_labeled_samples is not None
        else None
    )
    overview_df = pd.DataFrame(
        {
            "指标": [
                "样本数",
                "字段数",
                "数值列数",
                "类别列数",
                "目标列",
                "累计轮次数",
                "最新轮次",
                "当前分析轮次",
                "当前已标注样本数",
                "当前剩余候选池样本数",
            ],
            "取值": [
                analysis_df.shape[0],
                analysis_df.shape[1],
                len(numeric_columns),
                len(categorical_columns),
                effective_target_col or "未识别",
                total_rounds if total_rounds else "尚无轮次历史",
                latest_round_index if latest_round_index is not None else "尚未识别",
                current_round_index if current_round_index is not None else "尚未识别",
                current_labeled_samples,
                remaining_candidate_samples if remaining_candidate_samples is not None else "无法推断",
            ],
        }
    )
    display_table(overview_df, title="本次实验概览", rename_columns=False, precision=4, gradient_subset=[])

    column_profile_df = pd.DataFrame(
        {
            "字段名": analysis_df.columns,
            "数据类型": analysis_df.dtypes.astype(str).values,
            "缺失比例": analysis_df.isna().mean().values,
            "非空样本数": analysis_df.notna().sum().values,
        }
    ).sort_values(["缺失比例", "字段名"], ascending=[False, True])
    display_table(column_profile_df, title="字段类型与缺失概况", rename_columns=False, precision=4)

    print("这张表说明什么：它先回答“数据有多大、字段是什么类型、哪些列最容易出问题”。")

    plot_missingness(analysis_df)

    if numeric_columns:
        numeric_stats_df = analysis_df[numeric_columns].describe().T
        numeric_stats_df["skewness"] = analysis_df[numeric_columns].skew(numeric_only=True)
        display_table(numeric_stats_df, title="数值特征描述统计", rename_columns=False, precision=4)
        print("这张表说明什么：它概括了数值特征的中心位置、离散程度和偏态，是后续缩放与变换的依据。")

    plot_numeric_histograms(analysis_df, numeric_columns)
    plot_boxplots(analysis_df, numeric_columns)

    if numeric_columns:
        skew_df = (
            analysis_df[numeric_columns]
            .skew(numeric_only=True)
            .rename("偏度")
            .sort_values(key=np.abs, ascending=False)
            .reset_index()
            .rename(columns={"index": "特征"})
        )
        display_table(skew_df, title="偏态统计汇总", rename_columns=False, precision=4, gradient_subset=["偏度"])
        print("这张表说明什么：偏态绝对值越大，说明分布越不对称，模型训练时越可能受极端值影响。")

    plot_correlation_heatmap(analysis_df, numeric_columns)
    plot_categorical_distributions(analysis_df, categorical_columns)
    plot_target_distribution(analysis_df, effective_target_col)

    outlier_summary_df = summarize_outliers(analysis_df, numeric_columns)
    if not outlier_summary_df.empty:
        display_table(outlier_summary_df, title="异常值比例统计", rename_columns=False, precision=4, gradient_subset=["异常值比例"])
        print("这张表说明什么：它统计了每个数值特征的粗略异常值比例，便于判断哪些字段最值得重点清洗。")

    eda_summary = summarize_eda_issues(analysis_df, numeric_columns, effective_target_col)
    eda_conclusion_lines = []
    if eda_summary["high_missing_cols"]:
        eda_conclusion_lines.append(f"- 缺失严重的字段：{', '.join(eda_summary['high_missing_cols'])}")
    if eda_summary["high_outlier_cols"]:
        eda_conclusion_lines.append(f"- 异常值比例较高的字段：{', '.join(eda_summary['high_outlier_cols'])}")
    if eda_summary["high_skew_cols"]:
        eda_conclusion_lines.append(f"- 偏态明显的字段：{', '.join(eda_summary['high_skew_cols'][:10])}")
    if eda_summary["high_corr_pairs"]:
        preview_pairs = [f"{left} / {right} ({value:.2f})" for left, right, value in eda_summary["high_corr_pairs"][:8]]
        eda_conclusion_lines.append(f"- 高度相关的特征对：{'; '.join(preview_pairs)}")

    if not eda_conclusion_lines:
        eda_conclusion_lines.append("- 当前没有发现特别突出的缺失、偏态、异常值或共线性问题。")

    display(Markdown("### EDA 自动发现的问题\n" + "\n".join(eda_conclusion_lines)))
    display(Markdown("### 建模前建议做的预处理\n" + "\n".join(f"- {item}" for item in eda_summary["preprocess_suggestions"])))
    print("这一步说明什么：EDA 不只是画图，还要把会影响建模的风险点提前暴露出来。")


## 3. 模型训练得顺不顺？

这一节分两层看：

- `最终结果层`：先看主模型和辅助模型的 train/validation 指标
- `训练过程层`：如果服务器上保留了逐 epoch 的 history，再判断是否收敛、是否过拟合或欠拟合

如果 history 不存在，本 notebook 会清楚提示“只能看最终指标，不能看训练曲线”。


In [ ]:
# 这一个代码块负责整理训练摘要、绘制 history 曲线，并自动输出中文训练结论。
def training_summary_to_table(summary_payload: dict[str, Any]) -> pd.DataFrame:
    # 把 training_summary.json 统一整理成长表，便于画柱状图和对比主模型/辅助模型。
    if not isinstance(summary_payload, dict) or not summary_payload:
        return pd.DataFrame()

    rows = []
    for model_name in ["main_model", "aux_model"]:
        model_metrics = summary_payload.get(model_name)
        if not isinstance(model_metrics, dict):
            continue
        for metric_name, metric_value in model_metrics.items():
            rows.append(
                {
                    "模型": "主模型" if model_name == "main_model" else "辅助模型",
                    "原始模型键": model_name,
                    "指标名": metric_name,
                    "指标显示名": to_cn_label(metric_name),
                    "数值": metric_value,
                }
            )
    return pd.DataFrame(rows)

def pick_history_series(history: dict[str, list[float]], aliases: list[str]) -> list[float]:
    # 同一个指标常常有多个命名版本，这里按别名顺序自动兼容。
    for alias in aliases:
        if alias in history and history[alias]:
            return [float(item) for item in history[alias]]
    return []


def normalize_history(history: dict[str, list[float]]) -> dict[str, list[float]]:
    # 把 history 字典里的值尽量都转换成 float 列表，方便统一分析。
    normalized = {}
    if not isinstance(history, dict):
        return normalized
    for key, value in history.items():
        try:
            normalized[str(key)] = [float(item) for item in value]
        except Exception:
            continue
    return normalized


def analyze_training_history(history: dict[str, list[float]], stable_window: int = 5) -> dict[str, Any]:
    # 根据训练/验证曲线自动判断收敛、过拟合和欠拟合。
    history = normalize_history(history)
    train_loss = pick_history_series(history, ["train_loss", "loss"])
    val_loss = pick_history_series(history, ["val_loss", "valid_loss", "validation_loss"])

    result = {
        "has_history": bool(train_loss or val_loss),
        "is_converged": None,
        "possible_overfitting": None,
        "possible_underfitting": None,
        "best_epoch": None,
        "best_val_loss": None,
        "conclusion_lines": [],
    }

    if not train_loss and not val_loss:
        result["conclusion_lines"].append("当前没有逐 epoch history，只能根据最终指标判断训练结果，无法观察完整收敛过程。")
        return result

    if val_loss:
        best_idx = int(np.argmin(val_loss))
        result["best_epoch"] = best_idx + 1
        result["best_val_loss"] = float(val_loss[best_idx])
    elif train_loss:
        best_idx = int(np.argmin(train_loss))
        result["best_epoch"] = best_idx + 1
        result["best_val_loss"] = float(train_loss[best_idx])

    tail = val_loss[-stable_window:] if len(val_loss) >= stable_window else val_loss
    if tail:
        tail_range = max(tail) - min(tail)
        baseline = max(abs(np.mean(tail)), 1e-8)
        result["is_converged"] = tail_range / baseline < 0.05

    if train_loss and val_loss:
        min_val = min(val_loss)
        result["possible_overfitting"] = val_loss[-1] > min_val * 1.05 and train_loss[-1] < train_loss[0] * 0.8
        train_improvement = (train_loss[0] - train_loss[-1]) / max(abs(train_loss[0]), 1e-8)
        val_improvement = (val_loss[0] - val_loss[-1]) / max(abs(val_loss[0]), 1e-8)
        result["possible_underfitting"] = train_improvement < 0.10 and val_improvement < 0.10

    if result["is_converged"] is True:
        result["conclusion_lines"].append("验证集 loss 在最后若干个 epoch 已基本趋稳，模型看起来已经基本收敛。")
    elif result["is_converged"] is False:
        result["conclusion_lines"].append("验证集 loss 末段仍有明显波动，说明训练过程还没有完全稳定。")

    if result["possible_overfitting"] is True:
        result["conclusion_lines"].append("训练集 loss 持续下降但验证集 loss 偏离最优点，存在过拟合迹象。")
    if result["possible_underfitting"] is True:
        result["conclusion_lines"].append("训练集和验证集的改善都有限，存在欠拟合或训练不足的可能。")
    if result["best_epoch"] is not None:
        result["conclusion_lines"].append(f"验证集表现最好的 epoch 大约在第 {result['best_epoch']} 轮。")
    if not result["conclusion_lines"]:
        result["conclusion_lines"].append("训练过程整体平稳，但仍建议结合业务目标和最终误差指标一起判断。")
    return result


def plot_history_curves(history: dict[str, list[float]]) -> None:
    # 画 loss 曲线，并按存在的指标自动补充 RMSE/MAE/accuracy 曲线。
    history = normalize_history(history)
    metric_groups = [
        ("loss", ["train_loss", "loss"], ["val_loss", "valid_loss", "validation_loss"]),
        ("rmse", ["train_rmse", "rmse"], ["val_rmse", "valid_rmse", "validation_rmse"]),
        ("mae", ["train_mae", "mae"], ["val_mae", "valid_mae", "validation_mae"]),
        ("accuracy", ["train_acc", "acc", "accuracy"], ["val_acc", "val_accuracy", "valid_accuracy"]),
    ]

    plotted = False
    for metric_label, train_aliases, val_aliases in metric_groups:
        train_series = pick_history_series(history, train_aliases)
        val_series = pick_history_series(history, val_aliases)
        if not train_series and not val_series:
            continue

        plotted = True
        fig, ax = plt.subplots(figsize=(8.4, 5.1))
        epochs = np.arange(1, max(len(train_series), len(val_series)) + 1)
        if train_series:
            ax.plot(
                epochs[: len(train_series)],
                train_series,
                marker="o",
                markersize=4.5,
                linewidth=2.1,
                color=REPORT_COLORS["navy"],
                label=f"训练集{to_cn_label(metric_label)}",
            )
        if val_series:
            ax.plot(
                epochs[: len(val_series)],
                val_series,
                marker="s",
                markersize=4.5,
                linewidth=2.1,
                color=REPORT_COLORS["orange"],
                label=f"验证集{to_cn_label(metric_label)}",
            )
        style_axis(ax, f"{to_cn_label(metric_label)}训练曲线", "训练轮次", to_cn_label(metric_label))
        ax.legend(loc="best")
        finalize_figure(fig)
        plt.show()
        print(f"这张图说明什么：它用来观察{to_cn_label(metric_label)}是否稳定变化，以及训练集与验证集之间是否逐渐分叉。")

    if not plotted:
        print("history 文件已找到，但没有识别出可绘制的 loss / rmse / mae / accuracy 字段。")

training_metrics_df = training_summary_to_table(training_summary)
history_analysis = analyze_training_history(history_payload)

if training_metrics_df.empty:
    print("未检测到 training_summary.json 或内容为空，无法展示最终训练指标。")
else:
    display_table(training_metrics_df, title="训练指标汇总", rename_columns=False, precision=6, gradient_subset=["数值"])
    print("这张表说明什么：它把主模型和辅助模型的训练集/验证集指标放在一起，便于直接横向比较。")
    plot_df = training_metrics_df.copy()
    fig, ax = plt.subplots(figsize=(12.4, 6.0))
    sns.barplot(
        data=plot_df,
        x="指标显示名",
        y="数值",
        hue="模型",
        palette=[REPORT_COLORS["navy"], REPORT_COLORS["orange"]],
        ax=ax,
    )
    style_axis(ax, "主模型与辅助模型训练指标对比", "指标名称", "指标数值")
    tidy_tick_labels(ax, axis="x", rotation=28, width=9, ha="right")
    ax.legend(title="模型类型", loc="upper right")
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：柱状图能帮助快速判断验证集误差是否显著高于训练集，以及主模型和辅助模型谁更稳定。")
history_reason = None
if isinstance(history_raw_payload, dict) and history_raw_payload:
    history_reason = history_raw_payload.get("reason")

if history_payload:
    plot_history_curves(history_payload)
elif OPTIONAL_PATHS["history"] is not None and OPTIONAL_PATHS["history"].exists():
    print(f"history 文件已找到，但当前没有可用的逐 epoch 指标：{history_reason or '未记录具体原因'}")
else:
    print("当前没有找到可用的 history 文件，因此只能分析最终指标，不能判断完整训练曲线。")

display(Markdown("### 自动训练结论\n" + "\n".join(f"- {item}" for item in history_analysis["conclusion_lines"])))

if not training_metrics_df.empty:
    main_validation_energy = training_summary.get("main_model", {}).get("validation_energy_rmse")
    main_validation_gradient = training_summary.get("main_model", {}).get("validation_gradient_rmse")
    main_validation_pcc = training_summary.get("main_model", {}).get("validation_energy_pcc")
    print("如何看这些指标：")
    print(f"- energy RMSE 越小越好，当前主模型验证集约为：{main_validation_energy}")
    print(f"- gradient RMSE 越小越好，当前主模型验证集约为：{main_validation_gradient}")
    print(f"- PCC 越接近 1 越好，当前主模型验证集约为：{main_validation_pcc}")
    print("- 验证集比训练集更重要，因为它更接近模型在新样本上的真实表现。")
    print("- 训练成功不等于主动学习已经收敛，后面还要结合 UQ 和高不确定性比例继续判断。")

training_findings = {
    "history_analysis": history_analysis,
    "training_metrics_df": training_metrics_df,
}


## 4. 模型预测得准不准？

这一节默认按 `回归任务` 分析，重点看：

- `MAE / MSE / RMSE / R²`
- 真实值 vs 预测值
- 残差图和残差分布
- 哪些样本误差最大
- 模型在高值区还是低值区更容易出错

如果没有逐样本预测文件，本节会退化为“只展示 training_summary 中已有的最终指标”。


In [ ]:
# 这一个代码块负责逐样本回归评估与误差分析。
def build_uncertainty_df(payload: dict[str, Any]) -> pd.DataFrame:
    # 把 uncertainty_latest.json 整理成表格，方便与预测结果按 sample_id 合并。
    if not isinstance(payload, dict):
        return pd.DataFrame()
    samples = payload.get("samples", [])
    if not isinstance(samples, list):
        return pd.DataFrame()
    frame = pd.DataFrame(samples)
    if not frame.empty and CONFIG["ID_COL"] not in frame.columns and "sample_id" in frame.columns:
        frame[CONFIG["ID_COL"]] = frame["sample_id"]
    return frame


def normalize_split_label(value: Any) -> str:
    text = str(value).strip().lower()
    return text if text else "overall"


def prepare_prediction_df(
    prediction_df: pd.DataFrame,
    analysis_frame: pd.DataFrame,
    uq_frame: pd.DataFrame,
) -> tuple[pd.DataFrame, str]:
    # 优先使用服务器侧逐样本预测文件；如果没有，再尝试从分析表中寻找 y_true / y_pred。
    candidate = pd.DataFrame()
    note = ""

    if not prediction_df.empty:
        candidate = prediction_df.copy()
        note = "当前使用服务器提供的逐样本预测文件。"
    elif (
        not analysis_frame.empty
        and CONFIG["TRUE_COL"] in analysis_frame.columns
        and CONFIG["PRED_COL"] in analysis_frame.columns
    ):
        base_cols = [CONFIG["ID_COL"], CONFIG["TRUE_COL"], CONFIG["PRED_COL"]]
        if "split" in analysis_frame.columns:
            base_cols.append("split")
        candidate = analysis_frame[base_cols].copy()
        note = "当前直接使用分析表中已有的真实值与预测值列。"

    if candidate.empty:
        return pd.DataFrame(), "未检测到逐样本预测结果，因此无法做散点图、残差图和误差分桶分析。"

    if CONFIG["ID_COL"] not in candidate.columns:
        candidate[CONFIG["ID_COL"]] = [f"sample_{idx:04d}" for idx in range(len(candidate))]

    if CONFIG["TRUE_COL"] not in candidate.columns or CONFIG["PRED_COL"] not in candidate.columns:
        return pd.DataFrame(), "预测结果表缺少真实值或预测值列，请检查 TRUE_COL / PRED_COL 设置。"

    candidate = candidate.copy()
    candidate[CONFIG["TRUE_COL"]] = pd.to_numeric(candidate[CONFIG["TRUE_COL"]], errors="coerce")
    candidate[CONFIG["PRED_COL"]] = pd.to_numeric(candidate[CONFIG["PRED_COL"]], errors="coerce")
    candidate["residual"] = candidate[CONFIG["PRED_COL"]] - candidate[CONFIG["TRUE_COL"]]
    candidate["abs_error"] = candidate["residual"].abs()
    if "split" in candidate.columns:
        candidate["split"] = candidate["split"].map(normalize_split_label)
        candidate["split_display"] = candidate["split"].map(lambda item: SPLIT_DISPLAY_MAP.get(item, to_cn_label(item)))

    if not uq_frame.empty and CONFIG["ID_COL"] in uq_frame.columns:
        candidate = candidate.merge(
            uq_frame[[col for col in uq_frame.columns if col in {CONFIG["ID_COL"], "uncertainty", "exceeds_threshold"}]],
            on=CONFIG["ID_COL"],
            how="left",
        )

    return candidate.dropna(subset=[CONFIG["TRUE_COL"], CONFIG["PRED_COL"]]), note


def evaluate_regression(frame: pd.DataFrame) -> dict[str, float]:
    # 汇总常用回归指标。
    if frame.empty:
        return {}
    y_true = frame[CONFIG["TRUE_COL"]].to_numpy()
    y_pred = frame[CONFIG["PRED_COL"]].to_numpy()
    mse = mean_squared_error(y_true, y_pred)
    metrics = {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MSE": float(mse),
        "RMSE": float(np.sqrt(mse)),
    }
    try:
        metrics["R2"] = float(r2_score(y_true, y_pred))
    except Exception:
        metrics["R2"] = float("nan")
    return metrics


def evaluate_regression_by_split(frame: pd.DataFrame) -> pd.DataFrame:
    # 如果预测表里有 split 列，就分别计算 subtrain / validation / overall 的指标。
    if frame.empty:
        return pd.DataFrame()

    rows = [
        {
            "split": "overall",
            "split_display": SPLIT_DISPLAY_MAP["overall"],
            "sample_count": len(frame),
            **evaluate_regression(frame),
        }
    ]

    if "split" in frame.columns:
        for split_name, split_frame in frame.groupby("split", dropna=False):
            metrics = evaluate_regression(split_frame)
            if not metrics:
                continue
            rows.append(
                {
                    "split": split_name,
                    "split_display": SPLIT_DISPLAY_MAP.get(split_name, to_cn_label(split_name)),
                    "sample_count": len(split_frame),
                    **metrics,
                }
            )

    return pd.DataFrame(rows)


def plot_split_regression_metrics(metrics_frame: pd.DataFrame) -> None:
    # 把总体、训练子集和验证子集的指标拆开画，避免把它们混成一团。
    if metrics_frame.empty or metrics_frame["split"].nunique() <= 1:
        return

    plot_frame = metrics_frame.copy()
    metric_names = ["MAE", "RMSE", "R2"]
    fig, axes = create_subplots(1, len(metric_names), kind="wide", width_scale=1.05)
    axes = np.array(axes).reshape(-1)
    for idx, (ax, metric_name) in enumerate(zip(axes, metric_names)):
        ax.bar(
            plot_frame["split_display"],
            plot_frame[metric_name],
            color=build_palette(len(plot_frame)),
            edgecolor="white",
            linewidth=0.8,
        )
        style_axis(ax, f"{to_cn_label(metric_name)} 分 split 对比", "数据切分", to_cn_label(metric_name))
        tidy_tick_labels(ax, axis="x", rotation=18, width=8, ha="right")
        add_bar_labels(ax, fmt="{:.4f}" if metric_name != "R2" else "{:.3f}")
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：它把训练子集、验证子集和总体的回归指标拆开展示，能一眼看出是否存在训练好看但验证变差的情况。")


def plot_true_vs_pred(frame: pd.DataFrame) -> None:
    # 理想情况下，散点应该尽量贴近 y=x 参考线。
    if frame.empty:
        return
    y_true = frame[CONFIG["TRUE_COL"]]
    y_pred = frame[CONFIG["PRED_COL"]]
    line_min = min(y_true.min(), y_pred.min())
    line_max = max(y_true.max(), y_pred.max())
    fig, ax = create_figure("square")
    scatter = ax.scatter(
        y_true,
        y_pred,
        c=frame["abs_error"],
        cmap=REPORT_HEATMAP_CMAP,
        s=76,
        alpha=0.88,
        edgecolor="white",
        linewidth=0.7,
    )
    ax.plot(
        [line_min, line_max],
        [line_min, line_max],
        linestyle="--",
        color=REPORT_COLORS["slate"],
        linewidth=1.4,
        label="理想参考线 y=x",
    )
    ax.set_aspect("equal", adjustable="box")
    style_axis(ax, "真实值与预测值对照", to_cn_label(CONFIG["TRUE_COL"]), to_cn_label(CONFIG["PRED_COL"]))
    cbar = fig.colorbar(scatter, ax=ax, shrink=0.85)
    cbar.set_label("绝对误差", color=REPORT_COLORS["ink"])
    ax.legend(loc="upper left")
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：散点越贴近参考线，说明模型整体预测越准确；颜色越深的位置表示误差更大。")


def plot_residuals(frame: pd.DataFrame) -> None:
    # 残差图常用来检查误差是否随预测值系统性变化。
    if frame.empty:
        return
    fig, ax = create_figure("wide")
    ax.scatter(
        frame[CONFIG["PRED_COL"]],
        frame["residual"],
        c=frame["abs_error"],
        cmap=REPORT_DIVERGING_CMAP,
        s=68,
        alpha=0.82,
        edgecolor="white",
        linewidth=0.65,
    )
    ax.axhline(0, linestyle="--", color=REPORT_COLORS["slate"], linewidth=1.3)
    style_axis(ax, "残差趋势图", to_cn_label(CONFIG["PRED_COL"]), "残差（预测值 - 真实值）")
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：如果残差在 0 附近随机散开，说明误差模式较健康；若出现趋势，则可能有系统性偏差。")


def plot_residual_distribution(frame: pd.DataFrame) -> None:
    # 残差分布图帮助判断误差是否偏斜、是否存在长尾。
    if frame.empty:
        return
    fig, ax = create_figure("standard")
    sns.histplot(
        frame["residual"].dropna(),
        kde=True,
        color=REPORT_COLORS["gold"],
        ax=ax,
        bins=min(28, max(12, int(np.sqrt(len(frame) or 1)))),
    )
    ax.axvline(0, linestyle="--", linewidth=1.3, color=REPORT_COLORS["slate"])
    style_axis(ax, "残差分布图", "残差（预测值 - 真实值）", "样本频数")
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：残差越集中在 0 附近越好；若明显偏移或长尾，说明模型在某些区间误差较大。")


def error_by_true_value_bins(frame: pd.DataFrame, n_bins: int = 5) -> pd.DataFrame:
    # 按真实值区间统计误差，帮助判断高值区或低值区谁更难预测。
    if frame.empty or frame[CONFIG["TRUE_COL"]].nunique() < 2:
        return pd.DataFrame()
    temp = frame.copy()
    try:
        temp["true_value_bin"] = pd.qcut(temp[CONFIG["TRUE_COL"]], q=min(n_bins, temp[CONFIG["TRUE_COL"]].nunique()), duplicates="drop")
    except Exception:
        return pd.DataFrame()
    return (
        temp.groupby("true_value_bin", observed=False)
        .agg(
            sample_count=(CONFIG["ID_COL"], "count"),
            mean_abs_error=("abs_error", "mean"),
            bin_rmse=("residual", lambda x: float(np.sqrt(np.mean(np.square(x))))),
        )
        .reset_index()
    )


uq_df = build_uncertainty_df(uncertainty_payload)
prediction_eval_df, prediction_note = prepare_prediction_df(external_prediction_df, analysis_df, uq_df)
regression_metrics = evaluate_regression(prediction_eval_df)
split_metrics_df = evaluate_regression_by_split(prediction_eval_df)

print(prediction_note)
if prediction_eval_df.empty:
    if not training_metrics_df.empty:
        display_table(training_metrics_df, title="训练指标汇总", rename_columns=False, precision=6, gradient_subset=["数值"])
    print("提示：当前没有逐样本预测结果，所以这一节无法回答更细的误差模式问题。")
    regression_analysis = {
        "metrics": {},
        "split_metrics": pd.DataFrame(),
        "top_errors": pd.DataFrame(),
        "error_bins": pd.DataFrame(),
        "conclusion_lines": ["请在服务器上提供逐样本预测文件，以启用真实值 vs 预测值、残差图和误差分桶分析。"],
    }
else:
    metrics_df = pd.DataFrame(
        {"指标": [to_cn_label(metric) for metric in regression_metrics.keys()], "数值": list(regression_metrics.values())}
    )
    display_table(metrics_df, title="回归评估指标（总体）", rename_columns=False, precision=6, gradient_subset=["数值"])
    print("这张表说明什么：它用 MAE、MSE、RMSE、R² 四个角度概括模型整体拟合质量。")

    if not split_metrics_df.empty:
        split_display_df = split_metrics_df.rename(
            columns={
                "split_display": "数据切分",
                "sample_count": "样本数",
                "MAE": "MAE",
                "MSE": "MSE",
                "RMSE": "RMSE",
                "R2": "R²",
            }
        )
        display_table(split_display_df, title="分 split 的回归评估指标", rename_columns=False, precision=6, gradient_subset=["MAE", "RMSE", "R²"])
        print("这张表说明什么：它把训练子集、验证子集和总体分开列出，便于判断有没有把训练集成绩和验证集成绩混在一起。")
        plot_split_regression_metrics(split_metrics_df)

    plot_true_vs_pred(prediction_eval_df)
    plot_residuals(prediction_eval_df)
    plot_residual_distribution(prediction_eval_df)

    top_error_df = prediction_eval_df.sort_values("abs_error", ascending=False).head(CONFIG["TOP_N_ERROR"])
    display_table(top_error_df, title="误差最大的样本", rename_columns=True, precision=6)
    print("这张表说明什么：它列出了误差最大的样本，适合回查几何、标注或模型是否在某些结构上持续失效。")

    error_bin_df = error_by_true_value_bins(prediction_eval_df)
    if not error_bin_df.empty:
        display_df = error_bin_df.rename(
            columns={
                "true_value_bin": "真实值区间",
                "sample_count": "样本数",
                "mean_abs_error": "平均绝对误差",
                "bin_rmse": "区间 RMSE",
            }
        )
        display_table(display_df, title="按真实值区间统计误差", rename_columns=False, precision=4, gradient_subset=["平均绝对误差", "区间 RMSE"])
        fig, ax = create_figure("wide")
        sns.barplot(data=display_df, x="真实值区间", y="平均绝对误差", color=REPORT_COLORS["blue"], ax=ax)
        style_axis(ax, "按真实值区间统计的平均绝对误差", "真实值区间", "平均绝对误差")
        tidy_tick_labels(ax, axis="x", rotation=20, width=12, ha="right")
        add_bar_labels(ax, fmt="{:.3f}")
        finalize_figure(fig)
        plt.show()
        print("这张图说明什么：它用于判断模型在高值区还是低值区更容易出错。")
    else:
        top_error_df = prediction_eval_df.sort_values("abs_error", ascending=False).head(CONFIG["TOP_N_ERROR"])

    conclusion_lines = [
        f"模型总体 MAE 为 {regression_metrics.get('MAE', float('nan')):.6f}。",
        f"模型总体 RMSE 为 {regression_metrics.get('RMSE', float('nan')):.6f}。",
        f"模型总体 R² 为 {regression_metrics.get('R2', float('nan')):.6f}。",
        "当前没有独立测试集；这里的训练子集 / 验证子集来自已标注样本内部的随机切分。",
    ]

    if not split_metrics_df.empty and {"subtrain", "validation"}.issubset(set(split_metrics_df["split"])):
        subtrain_row = split_metrics_df.loc[split_metrics_df["split"] == "subtrain"].iloc[0]
        validation_row = split_metrics_df.loc[split_metrics_df["split"] == "validation"].iloc[0]
        conclusion_lines.append(
            f"训练子集 RMSE 为 {subtrain_row['RMSE']:.6f}，验证子集 RMSE 为 {validation_row['RMSE']:.6f}。"
        )
        if validation_row["RMSE"] > subtrain_row["RMSE"] * 1.20:
            conclusion_lines.append("验证子集误差明显高于训练子集，要警惕过拟合。")
        else:
            conclusion_lines.append("训练子集与验证子集误差差距不大，当前没有特别强的过拟合信号。")

    if not error_bin_df.empty:
        worst_row = error_bin_df.sort_values("mean_abs_error", ascending=False).iloc[0]
        best_row = error_bin_df.sort_values("mean_abs_error", ascending=True).iloc[0]
        conclusion_lines.append(f"误差最高的真实值区间是 {worst_row['true_value_bin']}。")
        conclusion_lines.append(f"误差最低的真实值区间是 {best_row['true_value_bin']}。")

    residual_corr = prediction_eval_df[[CONFIG["PRED_COL"], "residual"]].corr().iloc[0, 1]
    if pd.notna(residual_corr) and abs(residual_corr) >= 0.3:
        conclusion_lines.append("残差与预测值存在一定相关性，说明模型可能仍有系统性偏差。")
    else:
        conclusion_lines.append("残差与预测值没有表现出特别强的线性趋势，误差模式相对更随机。")

    display(Markdown("### 回归评估结论\n" + "\n".join(f"- {item}" for item in conclusion_lines)))
    regression_analysis = {
        "metrics": regression_metrics,
        "split_metrics": split_metrics_df,
        "top_errors": top_error_df,
        "error_bins": error_bin_df,
        "conclusion_lines": conclusion_lines,
    }


## 5. 模型为什么这样预测？

这一节分成两个分支：

- `ADL / ANI 主线`：优先分析描述符、误差和不确定性之间的关系，而不是强行套树模型特征重要性
- `树模型 / 表格特征分支`：如果你在服务器上提供了可解释模型，则进一步画 feature importance、permutation importance 和 SHAP

也就是说，这一节不是默认假设“任何模型都有 feature_importances_”，而是根据模型类型走不同路径。


In [ ]:
# 这一个代码块负责解释模型行为，并尽量兼容 ADL 主线与树模型扩展主线。
def choose_descriptor_columns(frame: pd.DataFrame) -> list[str]:
    # 这些字段更像“可解释描述符”，优先拿来观察目标值和误差随它们如何变化。
    preferred = [
        "E_baseline",
        "E_target",
        "delta_E",
        "|F_baseline|",
        "|F_target|",
        "|delta_F|",
        "num_atoms",
        "charge",
        "multiplicity",
        "mean_atomic_number",
        "coordinate_std",
    ]
    available = [column for column in preferred if column in frame.columns and pd.api.types.is_numeric_dtype(frame[column])]
    if available:
        return available[:4]
    numeric_cols, _ = detect_column_groups(frame)
    excluded = {CONFIG["TRUE_COL"], CONFIG["PRED_COL"], "residual", "abs_error"}
    return [column for column in numeric_cols if column not in excluded][:4]


def plot_descriptor_relationships(frame: pd.DataFrame, target_col: str | None) -> None:
    # 观察重要描述符与目标值/误差之间的关系，帮助解释模型在哪些区域更容易出错。
    if frame.empty:
        print("没有可用于解释的数据表，跳过描述符关系图。")
        return

    descriptor_cols = choose_descriptor_columns(frame)
    if not descriptor_cols:
        print("没有合适的数值描述符，跳过描述符关系图。")
        return

    y_column = "abs_error" if "abs_error" in frame.columns else target_col
    if y_column is None or y_column not in frame.columns:
        print("没有可用的目标列或误差列，跳过描述符关系图。")
        return

    n_rows = math.ceil(len(descriptor_cols) / 2)
    fig, axes = plt.subplots(n_rows, 2, figsize=(12, 4.5 * n_rows))
    axes = np.array(axes).reshape(-1)
    for idx, (ax, column) in enumerate(zip(axes, descriptor_cols)):
        ax.scatter(
            frame[column],
            frame[y_column],
            s=66,
            alpha=0.78,
            color=REPORT_PALETTE[idx % len(REPORT_PALETTE)],
            edgecolor="white",
            linewidth=0.65,
        )
        style_axis(ax, f"{to_cn_label(column)}与{to_cn_label(y_column)}的关系", to_cn_label(column), to_cn_label(y_column))
    for ax in axes[len(descriptor_cols):]:
        ax.axis("off")
    fig.suptitle("重要描述符与目标或误差的关系", x=0.05, y=1.02, ha="left", fontsize=15, fontweight="bold", color=REPORT_COLORS["ink"])
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：它帮助判断模型误差是否集中在某些结构特征或能量区间。")

def plot_uq_vs_error(frame: pd.DataFrame) -> str:
    # 比较不确定性和真实误差是否一致，是主动学习里非常关键的一步。
    required_cols = {"uncertainty", "abs_error"}
    if frame.empty or not required_cols.issubset(frame.columns):
        return "当前没有同时包含 uncertainty 和 abs_error 的样本表，无法判断 UQ 与真实误差是否一致。"

    fig, ax = plt.subplots(figsize=(8.2, 5.2))
    if "exceeds_threshold" in frame.columns:
        hue_series = frame["exceeds_threshold"].map({True: "高于阈值", False: "低于阈值"}).fillna("未标记")
        sns.scatterplot(
            x=frame["uncertainty"],
            y=frame["abs_error"],
            hue=hue_series,
            palette=[REPORT_COLORS["blue"], REPORT_COLORS["orange"], REPORT_COLORS["slate"]][: hue_series.nunique()],
            s=72,
            alpha=0.82,
            ax=ax,
        )
        ax.legend(title="阈值状态", loc="upper left")
    else:
        ax.scatter(
            frame["uncertainty"],
            frame["abs_error"],
            s=72,
            alpha=0.82,
            color=REPORT_COLORS["teal"],
            edgecolor="white",
            linewidth=0.65,
        )
    style_axis(ax, "不确定性与绝对误差关系图", "不确定性（UQ）", "绝对误差")
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：如果 UQ 与绝对误差正相关，说明当前的不确定性估计对主动学习选点是有意义的。")

    corr = frame[["uncertainty", "abs_error"]].corr().iloc[0, 1]
    if pd.isna(corr):
        return "UQ 与绝对误差的相关性无法计算。"
    if corr >= 0.5:
        return f"UQ 与绝对误差呈较明显正相关（相关系数约 {corr:.3f}），说明不确定性估计较有参考价值。"
    if corr >= 0.2:
        return f"UQ 与绝对误差存在一定正相关（相关系数约 {corr:.3f}），但仍建议结合人工检查。"
    return f"UQ 与绝对误差相关性较弱（相关系数约 {corr:.3f}），说明当前 UQ 对真实误差的指示作用有限。"

def compare_high_error_and_high_uq(frame: pd.DataFrame) -> pd.DataFrame:
    # 对比高误差样本和高 UQ 样本的交集，判断主动学习是否真的挑中了困难样本。
    if frame.empty or "abs_error" not in frame.columns or "uncertainty" not in frame.columns:
        return pd.DataFrame()
    error_rank = frame[[CONFIG["ID_COL"], "abs_error"]].sort_values("abs_error", ascending=False).head(CONFIG["TOP_N_ERROR"])
    uq_rank = frame[[CONFIG["ID_COL"], "uncertainty"]].sort_values("uncertainty", ascending=False).head(CONFIG["TOP_N_ERROR"])
    merged = error_rank.merge(uq_rank, on=CONFIG["ID_COL"], how="outer")
    return merged


def get_feature_matrix(frame: pd.DataFrame, target_col: str | None) -> tuple[pd.DataFrame, pd.Series | None]:
    # 为特征重要性和 permutation importance 准备数值特征矩阵。
    if frame.empty or target_col is None or target_col not in frame.columns:
        return pd.DataFrame(), None
    numeric_cols, _ = detect_column_groups(frame)
    feature_cols = [
        column
        for column in numeric_cols
        if column not in {target_col}
        and column not in {CONFIG["TRUE_COL"], CONFIG["PRED_COL"], "residual", "abs_error", "uncertainty"}
    ]
    if not feature_cols:
        return pd.DataFrame(), None
    clean = frame[feature_cols + [target_col]].dropna()
    if clean.empty:
        return pd.DataFrame(), None
    return clean[feature_cols], clean[target_col]


def compute_feature_importance(model: Any, x_frame: pd.DataFrame, y_series: pd.Series | None) -> pd.DataFrame:
    # 对 sklearn 风格模型优先读 feature_importances_，否则退回 permutation importance。
    if model is None or x_frame.empty:
        return pd.DataFrame()

    if hasattr(model, "feature_importances_"):
        return (
            pd.DataFrame({"特征": x_frame.columns, "重要性": model.feature_importances_})
            .sort_values("重要性", ascending=False)
            .reset_index(drop=True)
        )

    if y_series is None:
        return pd.DataFrame()

    try:
        perm = permutation_importance(
            model,
            x_frame,
            y_series,
            n_repeats=10,
            random_state=42,
            scoring="neg_root_mean_squared_error",
        )
        return (
            pd.DataFrame({"特征": x_frame.columns, "重要性": perm.importances_mean})
            .sort_values("重要性", ascending=False)
            .reset_index(drop=True)
        )
    except Exception as exc:  # noqa: BLE001
        print(f"permutation importance 计算失败：{exc}")
        return pd.DataFrame()


def plot_feature_importance(importance_df: pd.DataFrame) -> None:
    # 条形图展示最重要的前若干个特征。
    if importance_df.empty:
        print("没有可展示的特征重要性结果。")
        return
    top_df = importance_df.head(15).copy()
    top_df["特征显示"] = [to_cn_label(item) for item in top_df["特征"]]
    fig, ax = plt.subplots(figsize=(10.2, max(5.0, len(top_df) * 0.38)))
    ax.barh(top_df["特征显示"][::-1], top_df["重要性"][::-1], color=REPORT_COLORS["navy"], edgecolor="white", linewidth=0.8)
    style_axis(ax, "特征重要性条形图", "重要性得分", "特征名称")
    add_bar_labels(ax, fmt="{:.3f}")
    finalize_figure(fig)
    plt.show()
    print("这张图说明什么：重要性越高，说明该特征对模型输出的影响通常越大。")

def try_shap_summary(model: Any, x_frame: pd.DataFrame) -> str:
    # SHAP 是可选增强项，不可用时只给提示，不让 notebook 崩掉。
    if model is None or x_frame.empty:
        return "缺少模型或特征矩阵，跳过 SHAP 分析。"
    if not module_available("shap"):
        return "当前环境未安装 shap，已跳过 SHAP summary plot。"

    try:
        import shap

        sample_x = x_frame.sample(min(len(x_frame), 300), random_state=42)
        explainer = shap.Explainer(model, sample_x)
        shap_values = explainer(sample_x)
        shap.summary_plot(shap_values, sample_x, show=True)
        return "已完成 SHAP summary plot，可据此查看哪些特征在全局上最影响模型输出。"
    except Exception as exc:  # noqa: BLE001
        return f"SHAP 分析未成功执行：{exc}"


explain_frame = analysis_df.copy()
if not prediction_eval_df.empty and CONFIG["ID_COL"] in explain_frame.columns:
    merge_cols = [CONFIG["ID_COL"], CONFIG["TRUE_COL"], CONFIG["PRED_COL"], "residual", "abs_error"]
    if "uncertainty" in prediction_eval_df.columns:
        merge_cols.append("uncertainty")
    explain_frame = explain_frame.merge(
        prediction_eval_df[merge_cols].drop_duplicates(subset=[CONFIG["ID_COL"]]),
        on=CONFIG["ID_COL"],
        how="left",
    )
elif not prediction_eval_df.empty:
    explain_frame = prediction_eval_df.copy()

plot_descriptor_relationships(explain_frame, effective_target_col)
uq_alignment_message = plot_uq_vs_error(explain_frame)
high_error_uq_df = compare_high_error_and_high_uq(explain_frame)
if not high_error_uq_df.empty:
    display_table(high_error_uq_df, title="高误差样本与高 UQ 样本对照", rename_columns=True, precision=4)
    print("这张表说明什么：它用于对比“模型最容易错的样本”和“UQ 最高的样本”是否重合。")

external_model = load_external_model()
feature_matrix, feature_target = get_feature_matrix(explain_frame, effective_target_col)
importance_df = compute_feature_importance(external_model, feature_matrix, feature_target)
if not importance_df.empty:
    importance_display_df = importance_df.assign(特征=importance_df["特征"].map(to_cn_label))
    display_table(importance_display_df, title="特征重要性汇总", rename_columns=False, precision=4, gradient_subset=["重要性"])
    plot_feature_importance(importance_df)
elif external_model is None:
    print("未提供外部可解释模型，因此跳过特征重要性分支。")
else:
    print("已经提供外部模型，但当前无法稳定计算特征重要性，请检查模型接口或特征矩阵。")

shap_message = try_shap_summary(external_model, feature_matrix)

explanation_lines = [uq_alignment_message]
if not importance_df.empty:
    top_features = ", ".join(importance_df["特征"].head(5).tolist())
    explanation_lines.append(f"当前最重要的前几个特征大致是：{top_features}。")
else:
    explanation_lines.append("当前默认解释主线仍然是描述符-误差关系与 UQ 行为分析，而不是树模型特征重要性。")
explanation_lines.append(shap_message)

display(Markdown("### 解释性分析结论\n" + "\n".join(f"- {item}" for item in explanation_lines)))
explanation_analysis = {
    "uq_alignment_message": uq_alignment_message,
    "importance_df": importance_df,
    "shap_message": shap_message,
    "high_error_uq_df": high_error_uq_df,
    "conclusion_lines": explanation_lines,
}


## 6. 主动学习迭代历史与实验汇报结论区

这一节会先回答“现在总共跑了几轮、当前最新轮次是什么、每轮选了多少点、不确定性比例怎么变”，再自动汇总成适合汇报的中文结论。


In [ ]:
# 这一个代码块负责输出主动学习历史、汇报版总结、适用性说明和复用建议。
def build_current_round_table(selection_payload: dict[str, Any], round_index: int | None) -> pd.DataFrame:
    if not isinstance(selection_payload, dict) or not selection_payload:
        return pd.DataFrame()

    selected_ids = selection_payload.get("selected_sample_ids", []) or normalize_selected_ids(selection_payload)
    selected_count = selection_payload.get("selected_count")
    if selected_count is None:
        selected_count = len(selected_ids)

    rows = [
        {"指标": "当前分析轮次", "取值": round_index if round_index is not None else "未识别"},
        {"指标": "本轮新增样本数", "取值": selected_count},
        {"指标": "高不确定性比例", "取值": selection_payload.get("uncertain_ratio")},
        {"指标": "高不确定性样本数", "取值": selection_payload.get("num_uncertain_samples")},
        {"指标": "几何池样本数", "取值": selection_payload.get("num_pool_samples")},
        {"指标": "是否收敛", "取值": selection_payload.get("converged")},
    ]
    return pd.DataFrame(rows)


def build_selected_id_table(selection_payload: dict[str, Any]) -> pd.DataFrame:
    selected_ids = selection_payload.get("selected_sample_ids", []) or normalize_selected_ids(selection_payload)
    if not selected_ids:
        return pd.DataFrame()
    return pd.DataFrame({"样本编号": selected_ids, "序号": np.arange(1, len(selected_ids) + 1)})


def plot_round_history(frame: pd.DataFrame, convergence_ratio: float | None = None) -> None:
    # 把每轮选点数、高不确定性比例、累计已标注样本和收敛状态放到一组图里。
    if frame.empty:
        print("还没有可展示的轮次历史，跳过多轮主动学习汇总图。")
        return

    plot_frame = frame.sort_values("round_index").copy()
    fig, axes = create_subplots(2, 2, kind="hero", height_scale=1.05)
    axes = np.array(axes).reshape(-1)

    axes[0].plot(
        plot_frame["round_index"],
        plot_frame["selected_count"],
        marker="o",
        color=REPORT_COLORS["navy"],
    )
    style_axis(axes[0], "每轮新增样本数", "轮次编号", "新增样本数")

    axes[1].plot(
        plot_frame["round_index"],
        plot_frame["uncertain_ratio"],
        marker="o",
        color=REPORT_COLORS["orange"],
    )
    if convergence_ratio is not None:
        axes[1].axhline(convergence_ratio, linestyle="--", linewidth=1.3, color=REPORT_COLORS["slate"], label="收敛阈值")
        axes[1].legend(loc="upper right")
    style_axis(axes[1], "每轮高不确定性比例", "轮次编号", "高不确定性比例")

    axes[2].plot(
        plot_frame["round_index"],
        plot_frame["labeled_samples_before_selection"],
        marker="o",
        color=REPORT_COLORS["teal"],
        label="选点前已标注样本数",
    )
    axes[2].plot(
        plot_frame["round_index"],
        plot_frame["expected_labeled_after_selection"],
        marker="s",
        linestyle="--",
        color=REPORT_COLORS["gold"],
        label="若完成该轮补标注后的样本数",
    )
    style_axis(axes[2], "累计样本数变化", "轮次编号", "样本数")
    axes[2].legend(loc="upper left")

    converged_numeric = plot_frame["converged"].map({True: 1, False: 0})
    axes[3].scatter(
        plot_frame["round_index"],
        converged_numeric,
        s=100,
        color=[REPORT_COLORS["teal"] if value else REPORT_COLORS["rose"] for value in converged_numeric.fillna(0)],
        edgecolor="white",
        linewidth=0.8,
    )
    axes[3].set_yticks([0, 1])
    axes[3].set_yticklabels(["未收敛", "已收敛"])
    style_axis(axes[3], "每轮收敛状态", "轮次编号", "收敛状态")

    finalize_figure(fig)
    plt.show()
    print("这组图说明什么：它把主动学习每一轮到底选了多少点、不确定性比例如何变化、累计样本如何增长以及是否达到收敛阈值，放在一张面板里统一观察。")


history_title = "主动学习迭代历史"
display(Markdown(f"### {history_title}"))

if round_history_df.empty:
    print("当前还没有 round_*_selection_summary.json，因此无法展示主动学习历史表。")
    current_round_table = build_current_round_table(selection_summary, current_round_index)
    if not current_round_table.empty:
        display_table(current_round_table, title="当前轮次选点分析", rename_columns=False, precision=4, gradient_subset=[])
else:
    current_round_table = build_current_round_table(selection_summary, current_round_index)
    if not current_round_table.empty:
        display_table(current_round_table, title="当前轮次选点分析", rename_columns=False, precision=4, gradient_subset=[])
        print("这张表说明什么：它概括了当前分析轮次一共新选了多少个样本、UQ 比例多高以及是否已经满足收敛条件。")

    current_selected_df = build_selected_id_table(selection_summary)
    if not current_selected_df.empty:
        display_table(current_selected_df, title="当前轮次被选中的样本编号", rename_columns=False, precision=4, max_rows=40, gradient_subset=[])
        print("这张表说明什么：它列出了当前这一轮真正被选出来、准备进入下一轮补标注的样本。")

    history_display_df = round_history_df.rename(
        columns={
            "round_index": "轮次编号",
            "selected_count": "本轮新增样本数",
            "uncertain_ratio": "高不确定性比例",
            "converged": "是否收敛",
            "num_pool_samples": "几何池样本数",
            "num_uncertain_samples": "高不确定性样本数",
            "labeled_samples_before_selection": "该轮选点前已标注样本数",
            "expected_labeled_after_selection": "该轮选点后预期样本数",
            "remaining_candidate_samples": "该轮选点前剩余候选池样本数",
            "selected_manifest_exists": "是否存在选点 manifest",
            "selected_count_consistent": "选点数量是否一致",
        }
    )
    display_table(
        history_display_df,
        title="主动学习轮次历史汇总",
        rename_columns=False,
        precision=4,
        gradient_subset=["本轮新增样本数", "高不确定性比例", "该轮选点前已标注样本数"],
    )
    print("这张表说明什么：它把每一轮的新增样本数、UQ 比例、候选池规模和收敛状态收拢到一起，方便直接做横向比较。")

    convergence_ratio = None
    if isinstance(base_config, dict):
        convergence_ratio = base_config.get("active_learning", {}).get("convergence_ratio")
    plot_round_history(round_history_df, convergence_ratio=convergence_ratio)




def build_formal_conclusions() -> list[str]:
    lines: list[str] = []

    if isinstance(regression_analysis, dict) and regression_analysis.get("metrics"):
        metrics = regression_analysis.get("metrics", {})
        lines.append(
            f"?????????????? MAE={metrics.get('MAE', float('nan')):.4f}?RMSE={metrics.get('RMSE', float('nan')):.4f}?R?={metrics.get('R2', float('nan')):.4f}?"
        )
    else:
        lines.append("????????????????????????????????????")

    split_frame = regression_analysis.get("split_metrics") if isinstance(regression_analysis, dict) else pd.DataFrame()
    if isinstance(split_frame, pd.DataFrame) and not split_frame.empty and {"split", "RMSE", "sample_count"}.issubset(split_frame.columns):
        if {"subtrain", "validation"}.issubset(set(split_frame["split"])):
            sub_row = split_frame.loc[split_frame["split"] == "subtrain"].iloc[0]
            val_row = split_frame.loc[split_frame["split"] == "validation"].iloc[0]
            lines.append(
                f"???/????????? RMSE={sub_row['RMSE']:.4f}????? RMSE={val_row['RMSE']:.4f}??????={int(val_row.get('sample_count', 0))}??"
            )
        else:
            lines.append("???/??????? split ???????????? subtrain/validation ???")
    else:
        lines.append("???/??????????? split ???")

    uq_payload = regression_analysis.get("uq_error_analysis", {}) if isinstance(regression_analysis, dict) else {}
    uq_lines = uq_payload.get("conclusion_lines", []) if isinstance(uq_payload, dict) else []
    if uq_lines:
        lines.append("?UQ ????????" + "?".join(uq_lines[:2]))
    else:
        lines.append("?UQ ??????????????????????????????")

    mandatory_missing = 0
    if isinstance(file_status_df, pd.DataFrame) and not file_status_df.empty and {"????", "????"}.issubset(file_status_df.columns):
        mandatory_missing = int((~file_status_df["????"] & file_status_df["????"]).sum())
    if mandatory_missing > 0:
        lines.append(f"??????????? {mandatory_missing} ???????????????????")

    lines.append("?????????????????????????????????????????????")
    return lines

experiment_summary_lines = []

if not analysis_df.empty:
    experiment_summary_lines.append(f"当前可分析数据共有 {analysis_df.shape[0]} 条样本、{analysis_df.shape[1]} 个字段。")
else:
    experiment_summary_lines.append("当前仓库没有可直接分析的数据表，建议在服务器环境补充数据后重新运行。")

if isinstance(base_config, dict) and base_config:
    pool_size = base_config.get("sampling", {}).get("default_pool_size")
    initial_points = base_config.get("active_learning", {}).get("initial_points")
    if pool_size is not None:
        experiment_summary_lines.append(f"配置中的几何池大小为 {pool_size}。")
    if initial_points is not None:
        experiment_summary_lines.append(f"配置中的初始训练样本数为 {initial_points}。")

if isinstance(delta_metadata, dict) and delta_metadata.get("num_samples") is not None:
    experiment_summary_lines.append(f"当前 delta 数据集记录的样本数为 {delta_metadata.get('num_samples')}。")

if total_rounds:
    experiment_summary_lines.append(f"当前已累计记录 {total_rounds} 轮主动学习选点。")
if latest_round_index is not None:
    experiment_summary_lines.append(f"最新轮次为第 {latest_round_index} 轮。")
if current_round_index is not None:
    experiment_summary_lines.append(f"当前 notebook 默认展示的是第 {current_round_index} 轮。")

if isinstance(training_diagnostics, dict) and training_diagnostics:
    experiment_summary_lines.append("训练诊断产物已导出到标准路径，可直接供 notebook 读取。")

if isinstance(pipeline_summary, dict) and pipeline_summary:
    experiment_summary_lines.append(f"最近一次主控流程 success = {pipeline_summary.get('success')}。")

if isinstance(train_main_status, dict) and "success" in train_main_status:
    experiment_summary_lines.append(f"主模型训练状态：{'成功' if train_main_status.get('success') else '失败'}。")
if isinstance(train_aux_status, dict) and "success" in train_aux_status:
    experiment_summary_lines.append(f"辅助模型训练状态：{'成功' if train_aux_status.get('success') else '失败'}。")

if history_analysis.get("conclusion_lines"):
    experiment_summary_lines.extend(history_analysis["conclusion_lines"][:3])

if regression_analysis.get("conclusion_lines"):
    experiment_summary_lines.extend(regression_analysis["conclusion_lines"][:4])

if explanation_analysis.get("conclusion_lines"):
    experiment_summary_lines.extend(explanation_analysis["conclusion_lines"][:3])

if isinstance(uncertainty_payload, dict) and uncertainty_payload.get("num_samples") is not None:
    experiment_summary_lines.append(f"本次 UQ 评估覆盖 {uncertainty_payload.get('num_samples')} 个样本。")

if isinstance(selection_summary, dict) and selection_summary:
    selected_count = selection_summary.get("selected_count")
    if selected_count is None:
        selected_count = len(selection_summary.get("selected_sample_ids", []) or normalize_selected_ids(selection_summary))
    uncertain_ratio = selection_summary.get("uncertain_ratio")
    converged = selection_summary.get("converged")
    experiment_summary_lines.append(f"当前分析轮次新增样本数为 {selected_count}。")
    if uncertain_ratio is not None:
        experiment_summary_lines.append(f"当前高不确定性比例约为 {uncertain_ratio:.2%}。")
    if converged is not None:
        experiment_summary_lines.append(f"当前是否满足主动学习收敛判据：{bool(converged)}。")

if not round_history_df.empty and len(round_history_df) >= 2:
    latest_ratio = round_history_df.iloc[-1]["uncertain_ratio"]
    prev_ratio = round_history_df.iloc[-2]["uncertain_ratio"]
    if pd.notna(latest_ratio) and pd.notna(prev_ratio):
        if latest_ratio < prev_ratio:
            experiment_summary_lines.append("与上一轮相比，高不确定性比例在下降。")
        elif latest_ratio > prev_ratio:
            experiment_summary_lines.append("与上一轮相比，高不确定性比例在上升，需要重点关注新一轮薄弱区域。")
        else:
            experiment_summary_lines.append("与上一轮相比，高不确定性比例基本持平。")

experiment_summary_lines.extend(build_formal_conclusions())

display(Markdown("### 面向实验汇报的自动结论\n" + "\n".join(f"- {item}" for item in experiment_summary_lines)))

applicability_df = pd.DataFrame(
    [
        {"类别": "可直接用", "内容": "多轮主动学习历史表与折线图", "说明": "自动识别最新轮次，并汇总每轮选点数、不确定性比例和收敛状态。"},
        {"类别": "可直接用", "内容": "EDA 全套", "说明": "包括缺失值、异常值、偏态、相关性、目标分布。"},
        {"类别": "可直接用", "内容": "总体 / 训练子集 / 验证子集回归指标", "说明": "前提是逐样本预测表里包含 split 列。"},
        {"类别": "可直接用", "内容": "真实值 vs 预测值、残差图、残差分布图", "说明": "默认作为回归主路径。"},
        {"类别": "可直接用", "内容": "history 自动结论", "说明": "前提是提供逐 epoch 训练记录。"},
        {"类别": "可选用", "内容": "accuracy 曲线", "说明": "只有 history 中存在 accuracy 字段时才绘制。"},
        {"类别": "可选用", "内容": "feature importance / permutation importance", "说明": "适合 sklearn 风格表格模型。"},
        {"类别": "可选用", "内容": "SHAP", "说明": "当前环境装有 shap 且模型兼容时再启用。"},
        {"类别": "这次不作为主路径", "内容": "分类专用指标与图", "说明": "当前 notebook 默认围绕回归任务组织。"},
        {"类别": "这次不作为主路径", "内容": "对 ANI 主模型直接套 feature_importances_", "说明": "当前 ADL 主线更适合看描述符、误差和 UQ 的关系。"},
    ]
)
display_table(applicability_df, title="适用性说明", rename_columns=False, precision=4, gradient_subset=[])
print("这张表说明什么：它明确区分了当前模板的主线能力、可选增强项和不建议强行套用的分析。")

reuse_df = pd.DataFrame(
    [
        {"优先修改项": "ROUND_SELECTION_MODE", "什么时候改": "想自动跟踪最新轮次或手动看旧轮次时", "说明": "默认 latest；若想看指定轮次，可配合 ROUND_INDEX_OVERRIDE 使用。"},
        {"优先修改项": "ROUND_INDEX_OVERRIDE", "什么时候改": "要回看某一轮时", "说明": "只有在 ROUND_SELECTION_MODE = manual 时才生效。"},
        {"优先修改项": "FEATURE_TABLE_PATH", "什么时候改": "服务器上有额外样本级特征表时", "说明": "优先提升 EDA 和解释性分析质量。"},
        {"优先修改项": "HISTORY_PATH", "什么时候改": "想画训练/验证 loss 曲线时", "说明": "标准路径下默认读取 train_main_history.json；若 history 缺失会自动降级。"},
        {"优先修改项": "PREDICTION_PATH", "什么时候改": "想切换到别的逐样本预测结果时", "说明": "标准路径下默认读取 train_main_predictions.csv。"},
        {"优先修改项": "MODEL_VIEW", "什么时候改": "想一键切换主模型/辅助模型分析时", "说明": "切到 aux 后，history 和 prediction 会自动改读辅助模型产物。"},
        {"优先修改项": "MODEL_PATH", "什么时候改": "需要做 feature importance 或 SHAP 时", "说明": "适合 sklearn 树模型或兼容解释器的模型。"},
        {"优先修改项": "TARGET_COL", "什么时候改": "目标不是 delta_E 时", "说明": "例如你要分析别的回归目标。"},
    ]
)
display_table(reuse_df, title="复用入口说明", rename_columns=False, precision=4, gradient_subset=[])
print("这张表说明什么：以后跑第二轮、第三轮或换体系时，通常优先改这里列出来的几个入口，不必重写整份 notebook。")
